# MYRIAD-Style CONUS Multi-Hazard Identification

This notebook builds single-hazard events and linked multi-hazard event sets for a selected CONUS analysis period.

## Canonical Production Path

1. Run Cell 1 to configure the analysis window, source contracts, hazard rules, and lag matrix.
2. Run Cell 2 to load utility helpers and file manifests.
3. Run the cells under **Canonical Production Pipeline** to construct masks, apply source-specific overrides, label events, enrich intensities, and write event/link outputs.
4. Run the cells under **Optional Development Diagnostics** only when validating event objects, source availability, spatial footprints, group structure, or rendering case-review artifacts.

The database preserves separate layers for individual hazard events, pairwise hazard links, and larger hazard groups. It also preserves separate temporal layers where the physical phenomenon requires them. This work is MYRIAD-informed and adapted for this project's source data and multi-risk objectives.

Wildfire uses filtered event polygons (`final_area_km2 >= 5`). Space weather uses the operational geoelectric-field definition. Hail uses MESH only as a direct radar-derived hail-size proxy when valid, and lightning is omitted when no usable lightning source is available. Missing or unusable source data are never interpreted as zero hazard.

In [ ]:
# MYRIAD-style multi-hazard identification for CONUS
# Pipeline configuration: edit only the analysis window and rules in this cell.

from __future__ import annotations

import re
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import xarray as xr
from scipy import ndimage
from scipy.spatial import cKDTree

try:
    import geopandas as gpd
    from shapely.geometry import Point
except Exception:
    gpd = None
    Point = None

ROOT = Path("/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data")
WILDFIRE_DIR = ROOT / "wildfire" / "daily"
TERRESTRIAL_DIR = ROOT / "terrestrial_weather"
SPACE_WEATHER_DIR = ROOT / "space_weather" / "NOAA_geoE"
GRID_PATH = ROOT / "analysis_grid_CONUS_50km.nc"
OUTPUT_ROOT = ROOT / "MYRIAD_events_output"
MONTHLY_OUTPUT_ROOT = OUTPUT_ROOT / "monthly"

# One-month pilot run. Change only these dates for future analysis periods.
TIME_START = "2024-05-01"
TIME_END = "2024-05-31"

run_start = pd.Timestamp(TIME_START)
run_end = pd.Timestamp(TIME_END)
if run_end < run_start:
    raise ValueError("TIME_END must be on or after TIME_START.")

RUN_LABEL = f"{run_start:%Y%m%d}_{run_end:%Y%m%d}"
OUTPUT_DIR = OUTPUT_ROOT / "diagnostics" / f"myriad_conus_{RUN_LABEL}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MONTHLY_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MULTIHAZARD_LAG_HOURS = 48.0
MIN_EVENT_DURATION_DAYS = 1
MIN_WILDFIRE_EVENT_AREA_KM2 = 5.0
ALLOW_SPATIAL_DRIFT_LINKING = True
USE_DASK = True
CHUNKS = {"time": 14}
DRY_RUN = False

HAZARD_RULES = {
    "wildfire": {"mode": "fixed_event_inventory", "min_area_km2": MIN_WILDFIRE_EVENT_AREA_KM2, "min_footprint_cells": 1},
    "heatwave": {"tail": "high", "percentile": 95, "persistence_days": 3, "min_footprint_cells": 20},
    "coldwave": {"tail": "low", "percentile": 5, "persistence_days": 3, "min_footprint_cells": 20},
    "extreme_heat": {"tail": "high", "percentile": 95, "persistence_steps": 3, "threshold_reference": "month", "min_footprint_cells": 20},
    "extreme_cold": {"tail": "low", "percentile": 5, "persistence_steps": 3, "threshold_reference": "month", "min_footprint_cells": 20},
    "extreme_wind": {"tail": "high", "percentile": 93, "persistence_steps": 6, "min_footprint_cells": 10},
    "extreme_precip": {"tail": "high", "percentile": 90, "source_variable": "MultiSensor_QPE_01H_Pass2_00.00", "min_footprint_cells": 20},
    "drought_proxy": {"tail": "low", "percentile": 15, "persistence_steps": 2, "min_footprint_cells": 20},
    "hail": {"mode": "screening", "tail": "high", "percentile": 90, "source_variable": "MAXSIZE", "source_description": "Maximum estimated hail size within a neighborhood; indirect severe-hail indicator", "source_units": "mm", "min_footprint_cells": 3},
    "lightning": {"mode": "screening", "tail": "high", "percentile": 90, "source_variable": "NLDN_CG_001min_AvgDensity_00.00", "source_description": "One-minute average cloud-to-ground lightning flash density", "source_units": "flashes km^-2 min^-1", "min_footprint_cells": 3},
    "space_weather_extreme": {"mode": "fixed_operational", "tail": "high", "percentile": 95, "aggregation_minutes": 5, "persistence_steps": 5, "min_footprint_cells": 10},
}

CONVECTIVE_PROXY_VARIABLES = {
    "echo_top_18dbz": {"variable": "EchoTop_18_00.50", "units": "km", "meaning": "Echo-top height at 18 dBZ; storm depth and convective intensity proxy"},
    "echo_top_50dbz": {"variable": "EchoTop_50_00.50", "units": "km", "meaning": "Echo-top height at 50 dBZ; strong storm core and severe-hail proxy"},
    "lightning_1min_density": {"variable": "NLDN_CG_001min_AvgDensity_00.00", "units": "flashes km^-2 min^-1", "meaning": "One-minute cloud-to-ground lightning flash density"},
    "lightning_15min_density": {"variable": "NLDN_CG_015min_AvgDensity_00.00", "units": "flashes km^-2 min^-1", "meaning": "Fifteen-minute cloud-to-ground lightning flash density"},
}

# Multi-hazard linkage lag, keyed by (leading_hazard, following_hazard).
PAIR_LAG_HOURS = {
    ("extreme_wind", "extreme_precip"): 24.0, ("extreme_precip", "extreme_wind"): 24.0,
    ("extreme_wind", "hail"): 24.0, ("hail", "extreme_wind"): 24.0,
    ("extreme_wind", "lightning"): 24.0, ("lightning", "extreme_wind"): 24.0,
    ("extreme_precip", "hail"): 24.0, ("hail", "extreme_precip"): 24.0,
    ("extreme_precip", "lightning"): 24.0, ("lightning", "extreme_precip"): 24.0,
    ("hail", "lightning"): 24.0, ("lightning", "hail"): 24.0,
    ("lightning", "wildfire"): 48.0, ("wildfire", "lightning"): 6.0,
    ("extreme_wind", "wildfire"): 12.0, ("wildfire", "extreme_wind"): 12.0,
    ("drought_proxy", "wildfire"): 24.0 * 30, ("wildfire", "drought_proxy"): 24.0 * 30,
    ("heatwave", "drought_proxy"): 24.0 * 14, ("drought_proxy", "heatwave"): 24.0,
    ("extreme_heat", "drought_proxy"): 24.0 * 14, ("drought_proxy", "extreme_heat"): 24.0,
    ("coldwave", "extreme_wind"): 48.0, ("extreme_wind", "coldwave"): 48.0,
    ("extreme_cold", "extreme_wind"): 48.0, ("extreme_wind", "extreme_cold"): 48.0,
}
SPACE_WEATHER_PAIR_LAG_HOURS = 24.0


def _lag_hours_for_ordered_pair(leading_hazard: str, following_hazard: str) -> float:
    if leading_hazard == "space_weather_extreme" or following_hazard == "space_weather_extreme":
        return SPACE_WEATHER_PAIR_LAG_HOURS
    return PAIR_LAG_HOURS.get((leading_hazard, following_hazard), MULTIHAZARD_LAG_HOURS)


THRESHOLDS = {
    "heatwave_pct": HAZARD_RULES["heatwave"]["percentile"],
    "coldwave_pct": HAZARD_RULES["coldwave"]["percentile"],
    "extreme_heat_pct": HAZARD_RULES["extreme_heat"]["percentile"],
    "extreme_cold_pct": HAZARD_RULES["extreme_cold"]["percentile"],
    "wind_pct": HAZARD_RULES["extreme_wind"]["percentile"],
    "precip_high_pct": HAZARD_RULES["extreme_precip"]["percentile"],
    "precip_low_pct": HAZARD_RULES["drought_proxy"]["percentile"],
    "hail_pct": HAZARD_RULES["hail"]["percentile"],
    "lightning_pct": HAZARD_RULES["lightning"]["percentile"],
    "wildfire_frp_pct": 90.0,
}
TEMPORAL_MODE = {"wildfire": "daily", "heatwave": "daily", "coldwave": "daily", "extreme_heat": "subdaily", "extreme_cold": "subdaily", "extreme_wind": "subdaily", "extreme_precip": "subdaily", "drought_proxy": "daily", "hail": "subdaily", "lightning": "subdaily", "space_weather_extreme": "subdaily"}
MIN_CONSECUTIVE_DAYS = {"heatwave": 3, "coldwave": 3, "drought_proxy": 2}
MIN_CONSECUTIVE_STEPS = {"extreme_heat": 3, "extreme_cold": 3, "extreme_wind": 6}
MIN_EVENT_FOOTPRINT_CELLS = {hazard: rule["min_footprint_cells"] for hazard, rule in HAZARD_RULES.items()}
MIN_EVENT_ACTIVE_CELL_STEPS = {"wildfire": 1, "heatwave": 6, "coldwave": 6, "extreme_heat": 12, "extreme_cold": 12, "extreme_wind": 4, "extreme_precip": 4, "drought_proxy": 10, "hail": 4, "lightning": 4, "space_weather_extreme": 24}
EVENT_END_GAP_STEPS = {"wildfire": 0, "heatwave": 1, "coldwave": 1, "extreme_heat": 1, "extreme_cold": 1, "extreme_wind": 1, "extreme_precip": 1, "drought_proxy": 2, "hail": 1, "lightning": 1, "space_weather_extreme": 2}
SPACE_WEATHER_OPERATIONAL = HAZARD_RULES["space_weather_extreme"]

print("Configured MYRIAD-CONUS pipeline")
print(f"  Time window:       {run_start:%Y-%m-%d} to {run_end:%Y-%m-%d}")
print(f"  Diagnostics:       {OUTPUT_DIR}")
print(f"  Monthly outputs:   {MONTHLY_OUTPUT_ROOT}")
print("  Hazards:           " + ", ".join(HAZARD_RULES))

In [ ]:
# Cell 2/5: utility helpers for loading files, inferring variables, and building hazard masks

DATE_PATTERNS = [
    re.compile(r"(20\d{2})(\d{2})(\d{2})"),
    re.compile(r"(20\d{2})[-_](\d{2})[-_](\d{2})"),
]


def _extract_date_from_name(name: str) -> Optional[pd.Timestamp]:
    for patt in DATE_PATTERNS:
        m = patt.search(name)
        if m:
            y, mo, d = m.groups()
            try:
                return pd.Timestamp(f"{y}-{mo}-{d}")
            except Exception:
                return None
    return None


def list_daily_files(data_dir: Path, glob_pattern: str) -> pd.DataFrame:
    rows = []
    for p in sorted(data_dir.glob(glob_pattern)):
        dt = _extract_date_from_name(p.name)
        if dt is None:
            continue
        rows.append({"time": dt.normalize(), "path": str(p)})
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out = out[(out["time"] >= pd.Timestamp(TIME_START)) & (out["time"] <= pd.Timestamp(TIME_END))]
    out = out.sort_values("time").drop_duplicates("time", keep="first").reset_index(drop=True)
    return out


def open_daily_stack(df_files: pd.DataFrame, chunks: Optional[dict] = None) -> xr.Dataset:
    if df_files.empty:
        raise FileNotFoundError("No daily files found for requested period.")
    paths = df_files["path"].tolist()
    open_kwargs = dict(combine="nested", concat_dim="time", decode_times=True)
    if chunks and USE_DASK:
        open_kwargs["chunks"] = chunks
    ds = xr.open_mfdataset(paths, **open_kwargs)

    if "time" in ds.coords and ds.sizes.get("time", 0) == len(df_files):
        ds = ds.assign_coords(time=df_files["time"].values)
    return ds


def infer_var(ds: xr.Dataset, candidates: Sequence[str], required: bool = True) -> Optional[str]:
    ds_vars = list(ds.data_vars)
    lower_map = {v.lower(): v for v in ds_vars}

    for c in candidates:
        if c in ds_vars:
            return c
    for c in candidates:
        c_low = c.lower()
        if c_low in lower_map:
            return lower_map[c_low]
    for c in candidates:
        c_low = c.lower()
        for v in ds_vars:
            if c_low in v.lower():
                return v

    if required:
        raise KeyError(f"None of candidate variables found: {candidates}")
    return None


def _rolling_consecutive(mask: xr.DataArray, min_days: int) -> xr.DataArray:
    if min_days <= 1:
        return mask
    rolling_hits = mask.astype(np.int16).rolling(time=min_days, min_periods=min_days).sum()
    core = rolling_hits >= min_days
    expanded = core.copy()
    for k in range(1, min_days):
        expanded = expanded | core.shift(time=k, fill_value=False)
    return expanded.fillna(False)


def _rolling_consecutive_steps(mask: xr.DataArray, min_steps: int) -> xr.DataArray:
    if min_steps <= 1:
        return mask.fillna(False)
    rolling_hits = mask.astype(np.int16).rolling(time=min_steps, min_periods=min_steps).sum()
    core = rolling_hits >= min_steps
    expanded = core.copy()
    for k in range(1, min_steps):
        expanded = expanded | core.shift(time=k, fill_value=False)
    return expanded.fillna(False)


def _quantile_threshold(da: xr.DataArray, q: float) -> xr.DataArray:
    if getattr(da, "chunks", None) is not None and "time" in da.dims:
        da = da.chunk({"time": -1})

    quant = da.quantile(q / 100.0, dim="time", skipna=True)
    if "quantile" in quant.dims:
        quant = quant.squeeze("quantile", drop=True)
    return quant


def _safe_to_numpy(da: xr.DataArray) -> np.ndarray:
    return np.asarray(da.fillna(False).astype(bool).values)


def _decode_time_coord_from_dataarray(da: xr.DataArray, fallback_base_date: str) -> pd.DatetimeIndex:
    """Robustly decode time coordinate for datasets with non-CF NOAA metadata.

    Handles patterns like:
      - UNITS='seconds since 2024-05-01T00:00:00.000'
      - REFTIME='2024-05-01T00:00:00.000'
      - Large numeric fill values (e.g., 9.969e36)
    """
    if "time" not in da.dims:
        return pd.DatetimeIndex([])

    t_da = da["time"]

    if np.issubdtype(t_da.dtype, np.datetime64):
        return pd.to_datetime(t_da.values, errors="coerce")

    vals = np.asarray(t_da.values)

    if np.issubdtype(vals.dtype, np.number):
        vals = vals.astype(float)
        vals[~np.isfinite(vals)] = np.nan

        # Drop obvious numeric fill values commonly used in NetCDF.
        huge = np.abs(vals) > 1e20
        if np.any(huge):
            vals[huge] = np.nan

        # Try CF-like decode using units attr (case-insensitive key/value handling).
        attrs_lc = {str(k).lower(): v for k, v in t_da.attrs.items()}
        units_attr = str(attrs_lc.get("units", "")).lower()
        reftime_attr = attrs_lc.get("reftime", None)

        if "since" in units_attr:
            unit_map = {
                "seconds": "s", "second": "s", "sec": "s",
                "minutes": "m", "minute": "m", "min": "m",
                "hours": "h", "hour": "h", "hr": "h",
                "days": "D", "day": "D",
                "milliseconds": "ms", "millisecond": "ms", "msec": "ms",
                "microseconds": "us", "microsecond": "us", "usec": "us",
                "nanoseconds": "ns", "nanosecond": "ns", "nsec": "ns",
            }
            parsed_unit = None
            for key, val in unit_map.items():
                if key in units_attr:
                    parsed_unit = val
                    break

            if parsed_unit is not None:
                try:
                    base = units_attr.split("since", 1)[1].strip()
                    origin = pd.Timestamp(base)
                    return pd.to_datetime(vals, unit=parsed_unit, origin=origin, errors="coerce")
                except Exception:
                    pass

        # NOAA fallback: numeric seconds offset from REFTIME.
        if reftime_attr is not None:
            try:
                base = pd.Timestamp(str(reftime_attr))
                return base + pd.to_timedelta(vals, unit="s")
            except Exception:
                pass

        # Generic UNIX-based heuristic.
        for unit in ("s", "ms", "us", "ns", "h", "m"):
            try:
                cand = pd.to_datetime(vals, unit=unit, origin="unix", errors="coerce")
                finite = cand[~pd.isna(cand)]
                if len(finite) > 0 and finite.min() >= pd.Timestamp("2000-01-01") and finite.max() <= pd.Timestamp("2100-01-01"):
                    return cand
            except Exception:
                continue

        # Numeric seconds-of-day fallback.
        finite_vals = vals[np.isfinite(vals)]
        if finite_vals.size > 0 and np.nanmin(finite_vals) >= 0 and np.nanmax(finite_vals) <= 172800:
            base = pd.Timestamp(fallback_base_date).normalize()
            return base + pd.to_timedelta(vals, unit="s")

    # Last resort.
    return pd.to_datetime(vals, errors="coerce")


def _safe_daily_time_coord(da: xr.DataArray) -> xr.DataArray:
    if "time" not in da.dims:
        return da

    t = _decode_time_coord_from_dataarray(da, fallback_base_date=TIME_START)
    valid = ~pd.isna(t)
    if not np.all(valid):
        da = da.isel(time=np.where(valid)[0])
        t = t[valid]

    da = da.assign_coords(time=pd.DatetimeIndex(t).normalize())
    if da.sizes.get("time", 0) > 0 and da.indexes["time"].has_duplicates:
        da = da.groupby("time").max(skipna=True)
    return da


def _safe_subdaily_time_coord(da: xr.DataArray) -> xr.DataArray:
    if "time" not in da.dims:
        return da

    t = _decode_time_coord_from_dataarray(da, fallback_base_date=TIME_START)
    valid = ~pd.isna(t)
    if not np.all(valid):
        da = da.isel(time=np.where(valid)[0])
        t = t[valid]

    t = pd.DatetimeIndex(t)

    # Some NOAA files encode time-of-day that resets each daily file.
    # If that pattern is detected, unfold segments into consecutive days
    # before duplicate-time collapse.
    if len(t) >= 100 and (t.max().normalize() - t.min().normalize()).days <= 2:
        resets = np.where(np.asarray(t[1:] < t[:-1]))[0] + 1
        if len(resets) > 0:
            seg_starts = np.r_[0, resets]
            seg_ends = np.r_[resets, len(t)]
            day0 = t[0].normalize()
            repaired = np.array(t.values, dtype="datetime64[ns]")

            for seg_i, (s0, s1) in enumerate(zip(seg_starts, seg_ends)):
                block = pd.DatetimeIndex(t[s0:s1])
                offsets = block - block.normalize()
                repaired[s0:s1] = (day0 + pd.to_timedelta(seg_i, unit="D") + offsets).values

            t = pd.DatetimeIndex(repaired)

    da = da.assign_coords(time=t)
    if da.sizes.get("time", 0) > 0 and da.indexes["time"].has_duplicates:
        da = da.groupby("time").max(skipna=True)
    return da


def _prepare_hazard_time(da: xr.DataArray, hazard_name: str) -> xr.DataArray:
    mode = TEMPORAL_MODE.get(hazard_name, "daily")
    if mode == "subdaily":
        return _safe_subdaily_time_coord(da)
    return _safe_daily_time_coord(da)


def _grid_space_weather_to_analysis_grid(ds_space_raw: xr.Dataset, ds_target_grid: xr.Dataset) -> xr.Dataset:
    """Map space-weather site data (1D gridpt/site) to analysis grid (y,x) using nearest-cell assignment."""
    if ds_space_raw is None:
        raise ValueError("ds_space_raw is None")

    ds_space = ds_space_raw.copy()

    # Decode NOAA time robustly before any manipulation.
    if "time" in ds_space.coords:
        fake_da = xr.DataArray(np.zeros(ds_space.sizes["time"]), dims=("time",), coords={"time": ds_space["time"]})
        t = _decode_time_coord_from_dataarray(fake_da, fallback_base_date=TIME_START)
        valid = ~pd.isna(t)
        if not np.all(valid):
            ds_space = ds_space.isel(time=np.where(valid)[0])
            t = t[valid]
        ds_space = ds_space.assign_coords(time=pd.DatetimeIndex(t))

    lat_name = "latitude" if ("latitude" in ds_space.coords or "latitude" in ds_space.data_vars) else "lat"
    lon_name = "longitude" if ("longitude" in ds_space.coords or "longitude" in ds_space.data_vars) else "lon"

    if lat_name in ds_space.data_vars:
        ds_space = ds_space.assign_coords({lat_name: ds_space[lat_name]})
    if lon_name in ds_space.data_vars:
        ds_space = ds_space.assign_coords({lon_name: ds_space[lon_name]})

    if lat_name not in ds_space.coords or lon_name not in ds_space.coords:
        raise ValueError("Space-weather dataset must include latitude/longitude as coords or data variables.")

    lat_da = ds_space[lat_name]
    lon_da = ds_space[lon_name]

    if len(lat_da.dims) == 1 and len(lon_da.dims) == 1 and lat_da.dims[0] == lon_da.dims[0]:
        site_dim = lat_da.dims[0]
        src_lat = lat_da.values.astype(float)
        src_lon = lon_da.values.astype(float)
    else:
        if "time" not in lat_da.dims or "time" not in lon_da.dims:
            raise ValueError("Space-weather latitude/longitude must be 1D site or 2D (time,site).")
        other_lat = [d for d in lat_da.dims if d != "time"]
        other_lon = [d for d in lon_da.dims if d != "time"]
        if len(other_lat) != 1 or len(other_lon) != 1 or other_lat[0] != other_lon[0]:
            raise ValueError("Could not infer space-weather site dimension from latitude/longitude.")

        site_dim = other_lat[0]
        lat_2d = lat_da.transpose("time", site_dim).values.astype(float)
        lon_2d = lon_da.transpose("time", site_dim).values.astype(float)
        src_lat = np.nanmedian(lat_2d, axis=0)
        src_lon = np.nanmedian(lon_2d, axis=0)

        bad = ~np.isfinite(src_lat) | ~np.isfinite(src_lon)
        if np.any(bad):
            valid_matrix = np.isfinite(lat_2d) & np.isfinite(lon_2d)
            for j in np.where(bad)[0]:
                rows = np.where(valid_matrix[:, j])[0]
                if len(rows) > 0:
                    r0 = rows[0]
                    src_lat[j] = lat_2d[r0, j]
                    src_lon[j] = lon_2d[r0, j]

        keep_sites = np.isfinite(src_lat) & np.isfinite(src_lon)
        if not keep_sites.any():
            raise ValueError("No finite space-weather site coordinates were available for gridding.")

        src_lat = src_lat[keep_sites]
        src_lon = src_lon[keep_sites]

    if "lat" not in ds_target_grid or "lon" not in ds_target_grid:
        raise ValueError("Target analysis grid must contain 2D lat/lon variables.")

    target_lat = ds_target_grid["lat"].values.astype(float).ravel()
    target_lon = ds_target_grid["lon"].values.astype(float).ravel()
    ny, nx = ds_target_grid.sizes["y"], ds_target_grid.sizes["x"]
    ncells = ny * nx

    tree = cKDTree(np.column_stack([target_lat, target_lon]))
    src_points = np.column_stack([src_lat, src_lon])
    _, nearest_idx = tree.query(src_points, k=1)
    nearest_idx = nearest_idx.astype(np.int64)

    out_vars = {}
    for var_name, da in ds_space.data_vars.items():
        if "time" not in da.dims or site_dim not in da.dims:
            continue

        arr = da.transpose("time", site_dim).values.astype(float)

        if "keep_sites" in locals() and arr.shape[1] != len(nearest_idx):
            arr = arr[:, keep_sites]

        nt = arr.shape[0]
        gridded = np.full((nt, ncells), np.nan, dtype=np.float32)

        for t_i in range(nt):
            row = arr[t_i]
            valid = np.isfinite(row)
            if not valid.any():
                continue
            sums = np.bincount(nearest_idx[valid], weights=row[valid], minlength=ncells)
            counts = np.bincount(nearest_idx[valid], minlength=ncells)
            mean_vals = np.full(ncells, np.nan, dtype=np.float32)
            nz = counts > 0
            mean_vals[nz] = (sums[nz] / counts[nz]).astype(np.float32)
            gridded[t_i, :] = mean_vals

        out_vars[var_name] = (("time", "y", "x"), gridded.reshape(nt, ny, nx))

    ds_out = xr.Dataset(
        data_vars=out_vars,
        coords={
            "time": ds_space["time"].values,
            "y": ds_target_grid["y"].values,
            "x": ds_target_grid["x"].values,
            "lat": (("y", "x"), ds_target_grid["lat"].values),
            "lon": (("y", "x"), ds_target_grid["lon"].values),
        },
        attrs={"gridding_method": "nearest_site_to_analysis_cell_mean"},
    )
    return ds_out


# Build file manifests
wildfire_files = list_daily_files(WILDFIRE_DIR, "fires_analysis_grid_*.nc")
terr_files = list_daily_files(TERRESTRIAL_DIR, "weather_CONUS_*.nc")
space_files = list_daily_files(SPACE_WEATHER_DIR, "*-empirical-EMTF-2022.12-2022.12.nc")

print("Daily file counts in time window")
print("  wildfire:", len(wildfire_files))
print("  terrestrial weather:", len(terr_files))
print("  space weather:", len(space_files))

## Optional Development Diagnostics

The following cells are not required for routine production generation. They validate source coverage, thresholds, event objects, spatial footprints, temporal continuity, group structure, and review artifacts.

In [ ]:
# Full-period input audit

AUDIT_START = pd.Timestamp("2024-01-01")
AUDIT_END = pd.Timestamp("2025-12-31")
expected_days = pd.date_range(AUDIT_START, AUDIT_END, freq="D")
AUDIT_DIR = OUTPUT_ROOT / "diagnostics" / "input_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)


def list_daily_files_for_window(data_dir, pattern, start, end):
    rows = []
    for path in sorted(data_dir.glob(pattern)):
        timestamp = _extract_date_from_name(path.name)
        if timestamp is not None and start <= timestamp <= end:
            rows.append({"time": timestamp.normalize(), "path": str(path)})
    return pd.DataFrame(rows)


def contiguous_date_ranges(days):
    days = pd.DatetimeIndex(days).sort_values()
    if len(days) == 0:
        return []
    starts = [days[0]]
    ends = []
    for previous, current in zip(days[:-1], days[1:]):
        if current - previous > pd.Timedelta(days=1):
            ends.append(previous)
            starts.append(current)
    ends.append(days[-1])
    return [{"start_date": start, "end_date": end, "n_days": int((end - start).days + 1)} for start, end in zip(starts, ends)]


def audit_daily_inventory(name, data_dir, pattern):
    found = list_daily_files_for_window(data_dir, pattern, AUDIT_START, AUDIT_END)
    found_days = pd.DatetimeIndex(found["time"]) if not found.empty else pd.DatetimeIndex([])
    duplicates = found[found.duplicated("time", keep=False)].sort_values("time") if not found.empty else found
    missing = expected_days.difference(found_days.unique())
    report = {
        "source": name,
        "expected_days": len(expected_days),
        "files_found": len(found),
        "unique_days": found_days.nunique(),
        "missing_days": len(missing),
        "duplicate_file_rows": len(duplicates),
        "first_available": found_days.min() if len(found_days) else pd.NaT,
        "last_available": found_days.max() if len(found_days) else pd.NaT,
    }
    return report, missing, duplicates, found


reports = []
missing_rows = []
duplicate_rows = []
source_specs = [
    ("wildfire", WILDFIRE_DIR, "fires_analysis_grid_*.nc"),
    ("terrestrial_weather", TERRESTRIAL_DIR, "weather_CONUS_*.nc"),
    ("space_weather", SPACE_WEATHER_DIR, "*-empirical-EMTF-2022.12-2022.12.nc"),
]
for name, directory, pattern in source_specs:
    report, missing, duplicates, inventory = audit_daily_inventory(name, directory, pattern)
    reports.append(report)
    missing_rows.extend({"source": name, "date": day} for day in missing)
    if not duplicates.empty:
        duplicate_rows.extend(duplicates.assign(source=name).to_dict("records"))

input_audit_summary = pd.DataFrame(reports)
input_audit_missing = pd.DataFrame(missing_rows)
input_audit_duplicates = pd.DataFrame(duplicate_rows)
input_audit_yearly = (
    pd.DataFrame({"date": expected_days})
    .assign(year=lambda frame: frame["date"].dt.year)
    .merge(
        input_audit_missing.assign(missing=True),
        how="cross",
    )
)

coverage_year_rows = []
for name, _, _ in source_specs:
    available = expected_days.difference(pd.DatetimeIndex(input_audit_missing.loc[input_audit_missing["source"] == name, "date"]))
    for year in [2024, 2025]:
        expected_year = expected_days[expected_days.year == year]
        coverage_year_rows.append({"source": name, "year": year, "expected_days": len(expected_year), "available_days": int((available.year == year).sum()), "missing_days": int((expected_year.difference(available)).size)})
input_audit_yearly = pd.DataFrame(coverage_year_rows)
input_audit_missing_ranges = pd.DataFrame(
    [
        {"source": name, **date_range}
        for name, _, _ in source_specs
        for date_range in contiguous_date_ranges(input_audit_missing.loc[input_audit_missing["source"] == name, "date"])
    ]
)

input_audit_summary.to_csv(AUDIT_DIR / "input_coverage_summary.csv", index=False)
input_audit_yearly.to_csv(AUDIT_DIR / "input_coverage_by_year.csv", index=False)
input_audit_missing.to_csv(AUDIT_DIR / "missing_dates.csv", index=False)
input_audit_missing_ranges.to_csv(AUDIT_DIR / "missing_date_ranges.csv", index=False)
input_audit_duplicates.to_csv(AUDIT_DIR / "duplicate_file_dates.csv", index=False)

representative_days = [pd.Timestamp("2024-01-13"), pd.Timestamp("2024-05-01"), pd.Timestamp("2024-07-24"), pd.Timestamp("2025-12-31")]
cadence_rows = []
for day in representative_days:
    terr_path = TERRESTRIAL_DIR / f"weather_CONUS_{day:%Y%m%d}.nc"
    space_candidates = sorted(SPACE_WEATHER_DIR.glob(f"{day:%Y%m%d}-empirical-EMTF-2022.12-2022.12.nc"))
    for source, path in [("terrestrial_weather", terr_path), ("space_weather", space_candidates[0] if space_candidates else None)]:
        if path is None or not path.exists():
            cadence_rows.append({"source": source, "date": day, "exists": False})
            continue
        with xr.open_dataset(path, decode_times=True) as dataset:
            if source == "terrestrial_weather":
                time = pd.DatetimeIndex(pd.to_datetime(dataset["time"].values, errors="coerce"))
                steps = np.diff(time.values) / np.timedelta64(1, "h") if len(time) > 1 else np.array([])
                variables_lower = {name.lower() for name in dataset.data_vars}
                cadence_rows.append({
                    "source": source,
                    "date": day,
                    "exists": True,
                    "time_samples": len(time),
                    "unique_time_samples": time.nunique(),
                    "start_time": time.min(),
                    "end_time": time.max(),
                    "median_step_hours": float(np.nanmedian(steps)) if len(steps) else np.nan,
                    "has_qpe_01h": "multisensor_qpe_01h_pass2_00.00" in variables_lower,
                    "temperature_variable": next((name for name in dataset.data_vars if name.lower() in {"t2m", "tmp2m", "temperature", "air_temperature", "tmean"}), None),
                })
            else:
                cadence_rows.append({"source": source, "date": day, "exists": True, "time_samples": dataset.sizes.get("time", 0), "time_decode": "handled by _decode_time_coord_from_dataarray"})

input_audit_cadence = pd.DataFrame(cadence_rows)
input_audit_cadence.to_csv(AUDIT_DIR / "representative_time_cadence.csv", index=False)
display(input_audit_summary)
display(input_audit_yearly)
display(input_audit_missing_ranges)
display(input_audit_cadence)
print(f"Saved input audit: {AUDIT_DIR}")

In [ ]:
# Runtime benchmark: terrestrial file loading + one quantile, extrapolated to the full 731-day period

import time

BENCHMARK_DIR = OUTPUT_ROOT / "diagnostics" / "runtime_benchmark"
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

benchmark_days_list = [14, 30, 60, 120]
bench_start = pd.Timestamp("2024-01-01")
benchmark_rows = []

for n_days in benchmark_days_list:
    bench_end = bench_start + pd.Timedelta(days=n_days - 1)
    files_n = list_daily_files_for_window(TERRESTRIAL_DIR, "weather_CONUS_*.nc", bench_start, bench_end)
    if len(files_n) < n_days:
        print(f"  Skipping {n_days} days: only {len(files_n)} files available.")
        continue

    t0 = time.perf_counter()
    ds_bench = open_daily_stack(files_n, chunks=CHUNKS)
    temp_var_bench = infer_var(ds_bench, ["T2M", "TMP2m", "temperature", "air_temperature", "tmean"], required=False)
    t_open = time.perf_counter() - t0

    t1 = time.perf_counter()
    if temp_var_bench is not None:
        temp_bench = _safe_subdaily_time_coord(ds_bench[temp_var_bench])
        if getattr(temp_bench, "chunks", None) is not None:
            temp_bench = temp_bench.chunk({"time": -1})
        _quantile_threshold(temp_bench, 95).compute()
    t_quantile = time.perf_counter() - t1

    ds_bench.close()
    benchmark_rows.append({"n_days": n_days, "n_files": len(files_n), "open_seconds": t_open, "quantile_seconds": t_quantile, "total_seconds": t_open + t_quantile})
    print(f"  {n_days} days ({len(files_n)} files): open={t_open:.2f}s, quantile={t_quantile:.2f}s")

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df.to_csv(BENCHMARK_DIR / "terrestrial_load_benchmark.csv", index=False)
display(benchmark_df)

if len(benchmark_df) >= 2:
    slope_total, intercept_total = np.polyfit(benchmark_df["n_days"], benchmark_df["total_seconds"], 1)
    est_731_seconds = slope_total * 731 + intercept_total
    print(f"\nLinear extrapolation to 731 days (one variable, load + quantile only): ~{est_731_seconds:.0f}s (~{est_731_seconds / 60:.1f} min)")
    print("This is a lower bound: full mask construction repeats this for wind/precip/hail/lightning,")
    print("plus event labeling and pairwise linking, which add substantial additional time at full scale.")

In [ ]:
# Threshold/mask diagnostics for known episodes (seasonal + spot-check windows)

import matplotlib.pyplot as plt

EVENT_WINDOWS = [
    {"name": "winter_cold_billings", "start": "2024-01-10", "end": "2024-01-16", "reference_time": pd.Timestamp("2024-01-13 12:00:00")},
    {"name": "ca_precip_feb", "start": "2024-02-03", "end": "2024-02-08", "reference_time": pd.Timestamp("2024-02-05 12:00:00")},
    {"name": "il_convection_march", "start": "2024-03-13", "end": "2024-03-15", "reference_time": pd.Timestamp("2024-03-14 21:00:00")},
    {"name": "heat_del_rio_late_may", "start": "2024-05-25", "end": "2024-05-30", "reference_time": pd.Timestamp("2024-05-28 18:00:00")},
    {"name": "heat_ne_maine_june", "start": "2024-06-17", "end": "2024-06-21", "reference_time": pd.Timestamp("2024-06-19 18:00:00")},
    {"name": "park_fire_onset_july", "start": "2024-07-22", "end": "2024-07-29", "reference_time": pd.Timestamp("2024-07-25 00:00:00")},
]

EVENT_WINDOW_DIR = OUTPUT_ROOT / "diagnostics" / "event_windows"
EVENT_WINDOW_DIR.mkdir(parents=True, exist_ok=True)


def build_window_hazard_masks(window_start, window_end):
    window_start = pd.Timestamp(window_start)
    window_end = pd.Timestamp(window_end)

    wildfire_files_w = list_daily_files_for_window(WILDFIRE_DIR, "fires_analysis_grid_*.nc", window_start, window_end)
    terr_files_w = list_daily_files_for_window(TERRESTRIAL_DIR, "weather_CONUS_*.nc", window_start, window_end)
    space_files_w = list_daily_files_for_window(SPACE_WEATHER_DIR, "*-empirical-EMTF-2022.12-2022.12.nc", window_start, window_end)

    result = {"masks": {}, "sources": {}, "coverage": {"terrestrial_days": len(terr_files_w), "space_days": len(space_files_w), "wildfire_days": len(wildfire_files_w)}}
    if terr_files_w.empty:
        return result

    ds_terr_w = open_daily_stack(terr_files_w, chunks=CHUNKS)
    common_time_w = pd.date_range(window_start, window_end, freq="D")

    temp_var_w = infer_var(ds_terr_w, ["T2M", "TMP2m", "temperature", "air_temperature", "tmean"], required=False)
    wind_var_w = infer_var(ds_terr_w, ["i10fg", "I10FG", "wind_gust", "gust", "MAX_SHEAR", "VII_00.50", "wind"], required=False)
    hail_var_w = infer_var(ds_terr_w, ["MESH_00.50", "MAXSIZE", "hail"], required=False)
    lightning_var_w = infer_var(ds_terr_w, ["FLASH_QPE_ARIMAX_00.00", "flash", "lightning"], required=False)
    precip_var_w = HAZARD_RULES["extreme_precip"]["source_variable"] if HAZARD_RULES["extreme_precip"]["source_variable"] in ds_terr_w.data_vars else None
    result["sources"] = {"temp_var": temp_var_w, "wind_var": wind_var_w, "hail_var": hail_var_w, "lightning_var": lightning_var_w, "precip_var": precip_var_w}

    if temp_var_w is not None:
        temp_hourly_w = _safe_subdaily_time_coord(ds_terr_w[temp_var_w])
        if getattr(temp_hourly_w, "chunks", None) is not None:
            temp_hourly_w = temp_hourly_w.chunk({"time": -1})
        heat_thr_w = _quantile_threshold(temp_hourly_w, THRESHOLDS["extreme_heat_pct"])
        cold_thr_w = _quantile_threshold(temp_hourly_w, THRESHOLDS["extreme_cold_pct"])
        result["masks"]["extreme_heat"] = _rolling_consecutive_steps((temp_hourly_w > heat_thr_w).fillna(False), MIN_CONSECUTIVE_STEPS["extreme_heat"])
        result["masks"]["extreme_cold"] = _rolling_consecutive_steps((temp_hourly_w < cold_thr_w).fillna(False), MIN_CONSECUTIVE_STEPS["extreme_cold"])
        result["fields"] = {"temp": temp_hourly_w}

    if wind_var_w is not None:
        wind_da_w = _prepare_hazard_time(ds_terr_w[wind_var_w], "extreme_wind")
        wind_thr_w = _quantile_threshold(wind_da_w, THRESHOLDS["wind_pct"])
        result["masks"]["extreme_wind"] = _rolling_consecutive_steps((wind_da_w > wind_thr_w).fillna(False), MIN_CONSECUTIVE_STEPS["extreme_wind"])
        result.setdefault("fields", {})["wind"] = wind_da_w

    if precip_var_w is not None:
        precip_da_w = _prepare_hazard_time(ds_terr_w[precip_var_w], "extreme_precip")
        precip_thr_w = _quantile_threshold(precip_da_w, THRESHOLDS["precip_high_pct"])
        result["masks"]["extreme_precip"] = (precip_da_w > precip_thr_w).fillna(False)
        result.setdefault("fields", {})["precip"] = precip_da_w

    if hail_var_w is not None:
        hail_da_w = _prepare_hazard_time(ds_terr_w[hail_var_w], "hail")
        hail_thr_w = _quantile_threshold(hail_da_w, THRESHOLDS["hail_pct"])
        result["masks"]["hail"] = (hail_da_w > hail_thr_w).fillna(False)

    if lightning_var_w is not None:
        li_da_w = _prepare_hazard_time(ds_terr_w[lightning_var_w], "lightning")
        li_thr_w = _quantile_threshold(li_da_w, THRESHOLDS["lightning_pct"])
        result["masks"]["lightning"] = (li_da_w > li_thr_w).fillna(False)

    if not space_files_w.empty:
        ds_space_raw_w = open_daily_stack(space_files_w, chunks=CHUNKS)
        ds_space_w = _grid_space_weather_to_analysis_grid(ds_space_raw_w, ds_target_grid)
        ex_var_w = infer_var(ds_space_w, ["Ex"], required=True)
        ey_var_w = infer_var(ds_space_w, ["Ey"], required=True)
        e_mag_w = _prepare_hazard_time(np.hypot(ds_space_w[ex_var_w], ds_space_w[ey_var_w]), "space_weather_extreme")
        sw_agg = int(SPACE_WEATHER_OPERATIONAL["aggregation_minutes"])
        if sw_agg > 1:
            e_mag_w = _prepare_hazard_time(e_mag_w.resample(time=f"{sw_agg}min").max(skipna=True), "space_weather_extreme")
        sw_thr_w = _quantile_threshold(e_mag_w, SPACE_WEATHER_OPERATIONAL["percentile"])
        sw_mask_w = _rolling_consecutive_steps((e_mag_w > sw_thr_w).fillna(False), SPACE_WEATHER_OPERATIONAL["persistence_steps"])
        result["masks"]["space_weather_extreme"] = sw_mask_w
        result.setdefault("fields", {})["space_weather"] = e_mag_w

    if wildfire_files_w.empty is False:
        wildfire_events_gpkg_w = ROOT / "wildfire" / "fire_events.gpkg"
        if gpd is not None and wildfire_events_gpkg_w.exists():
            events_gdf_w = gpd.read_file(wildfire_events_gpkg_w, layer="events")
            events_gdf_w["start_date"] = pd.to_datetime(events_gdf_w["start_date"], errors="coerce")
            events_gdf_w["end_date"] = pd.to_datetime(events_gdf_w["end_date"], errors="coerce")
            events_gdf_w["final_area_km2"] = pd.to_numeric(events_gdf_w["final_area_km2"], errors="coerce")
            overlapping = events_gdf_w[
                (events_gdf_w["final_area_km2"] >= MIN_WILDFIRE_EVENT_AREA_KM2)
                & (events_gdf_w["end_date"] >= window_start)
                & (events_gdf_w["start_date"] <= window_end)
            ]
            result["wildfire_events_overlapping"] = len(overlapping)

    return result


def active_fraction_series(mask_da):
    reduce_dims = [dim for dim in mask_da.dims if dim != "time"]
    series = mask_da.mean(dim=reduce_dims)
    return series.compute() if hasattr(series, "compute") else series


window_diagnostic_rows = []
for window in EVENT_WINDOWS:
    window_dir = EVENT_WINDOW_DIR / window["name"]
    window_dir.mkdir(parents=True, exist_ok=True)
    print(f"Building diagnostics for window: {window['name']} ({window['start']} to {window['end']})")
    window_result = build_window_hazard_masks(window["start"], window["end"])

    for hazard, mask_da in window_result.get("masks", {}).items():
        series = active_fraction_series(mask_da)
        series_df = pd.DataFrame({"time": pd.to_datetime(series["time"].values), "active_fraction": np.asarray(series.values, dtype=float)})
        series_df.to_csv(window_dir / f"{hazard}_active_fraction.csv", index=False)
        window_diagnostic_rows.append({
            "window": window["name"],
            "hazard": hazard,
            "n_timesteps": len(series_df),
            "max_active_fraction": float(series_df["active_fraction"].max()) if len(series_df) else np.nan,
            "mean_active_fraction": float(series_df["active_fraction"].mean()) if len(series_df) else np.nan,
        })

    reference_time = window["reference_time"]
    fields = window_result.get("fields", {})
    masks = window_result.get("masks", {})
    quicklook_specs = [
        ("temp", "extreme_heat", "Temperature (K) with extreme-heat mask"),
        ("temp", "extreme_cold", "Temperature (K) with extreme-cold mask"),
        ("precip", "extreme_precip", "1-hr precipitation with extreme mask"),
        ("wind", "extreme_wind", "Wind gust with extreme mask"),
        ("space_weather", "space_weather_extreme", "Geoelectric |E| with extreme mask"),
    ]
    n_panels = sum(1 for field_key, hazard_key, _ in quicklook_specs if field_key in fields and hazard_key in masks)
    if n_panels > 0:
        fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 4.5), squeeze=False)
        panel_index = 0
        for field_key, hazard_key, title in quicklook_specs:
            if field_key not in fields or hazard_key not in masks:
                continue
            field_da = fields[field_key]
            mask_da = masks[hazard_key]
            field_times = pd.DatetimeIndex(pd.to_datetime(field_da["time"].values))
            nearest_idx = int(np.argmin(np.abs((field_times - reference_time).total_seconds())))
            actual_time = field_times[nearest_idx]
            field_slice = np.asarray(field_da.isel(time=nearest_idx).values, dtype=float)
            mask_time = pd.DatetimeIndex(pd.to_datetime(mask_da["time"].values))
            mask_idx = int(np.argmin(np.abs((mask_time - actual_time).total_seconds())))
            mask_slice = np.asarray(mask_da.isel(time=mask_idx).fillna(False).values, dtype=bool)

            ax = axes[0, panel_index]
            lat_name = next((c for c in ["lat", "latitude", "y"] if c in field_da.coords), None)
            lon_name = next((c for c in ["lon", "longitude", "x"] if c in field_da.coords), None)
            if lat_name and lon_name and field_slice.ndim == 2:
                lat2d = np.asarray(field_da[lat_name].values)
                lon2d = np.asarray(field_da[lon_name].values)
                mesh = ax.pcolormesh(lon2d, lat2d, field_slice, shading="auto", cmap="viridis")
                fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.04)
                if mask_slice.shape == field_slice.shape and mask_slice.any():
                    ax.contour(lon2d, lat2d, mask_slice.astype(int), levels=[0.5], colors="red", linewidths=1.2)
            ax.set_title(f"{title}\n{actual_time:%Y-%m-%d %H:%M UTC}", fontsize=9)
            panel_index += 1

        fig.suptitle(f"Window: {window['name']}", fontsize=12)
        plt.tight_layout()
        out_png = window_dir / f"{window['name']}_quicklook.png"
        fig.savefig(out_png, dpi=140, bbox_inches="tight")
        plt.close(fig)
        print(f"  Saved quicklook: {out_png}")
    else:
        print(f"  No renderable fields/masks for window: {window['name']}")

window_diagnostics_summary = pd.DataFrame(window_diagnostic_rows)
window_diagnostics_summary.to_csv(EVENT_WINDOW_DIR / "window_diagnostics_summary.csv", index=False)
display(window_diagnostics_summary)
print(f"Saved event-window diagnostics: {EVENT_WINDOW_DIR}")

In [ ]:
# Event-filter attrition table

if "hazard_masks" not in globals() or not hazard_masks:
    raise RuntimeError("Run the hazard-mask construction cell before this diagnostic.")

ATTRITION_DIR = OUTPUT_ROOT / "diagnostics" / "event_attrition"
ATTRITION_DIR.mkdir(parents=True, exist_ok=True)


def attrition_for_hazard(mask_da, hazard_name):
    arr = _safe_to_numpy(mask_da)
    raw_components, n_raw = ndimage.label(arr, structure=_event_connectivity_structure(bool(ALLOW_SPATIAL_DRIFT_LINKING)))

    gap_steps = int(EVENT_END_GAP_STEPS.get(hazard_name, 0))
    bridged = _bridge_short_time_gaps(arr, gap_steps)
    bridged_components, n_bridged = ndimage.label(bridged, structure=_event_connectivity_structure(bool(ALLOW_SPATIAL_DRIFT_LINKING)))

    min_cells = int(MIN_EVENT_FOOTPRINT_CELLS.get(hazard_name, 1))
    min_active_steps = int(MIN_EVENT_ACTIVE_CELL_STEPS.get(hazard_name, 1))

    ny, nx = bridged.shape[1], bridged.shape[2]
    ncells = ny * nx
    ti, yi, xi = np.where(bridged_components > 0)
    if len(ti) == 0:
        return {"hazard": hazard_name, "raw_components": n_raw, "after_gap_bridge": n_bridged, "after_active_steps_filter": 0, "after_footprint_filter": 0, "final_retained": 0}

    lbl = bridged_components[ti, yi, xi].astype(np.int64)
    active_steps_counts = np.bincount(lbl, minlength=n_bridged + 1)
    flat_cell = yi.astype(np.int64) * nx + xi.astype(np.int64)
    unique_keys = np.unique(lbl * ncells + flat_cell)
    footprint_counts = np.bincount((unique_keys // ncells).astype(np.int64), minlength=n_bridged + 1)

    passed_active_steps = int(np.sum(active_steps_counts[1:] >= min_active_steps))
    passed_footprint = int(np.sum((active_steps_counts[1:] >= min_active_steps) & (footprint_counts[1:] >= min_cells)))

    final_tbl, _ = label_single_hazard_events(mask_da, hazard_name)
    return {
        "hazard": hazard_name,
        "raw_components": int(n_raw),
        "after_gap_bridge": int(n_bridged),
        "after_active_steps_filter": passed_active_steps,
        "after_footprint_filter": passed_footprint,
        "final_retained": int(len(final_tbl)),
    }


attrition_rows = [attrition_for_hazard(mask_da, hazard) for hazard, mask_da in hazard_masks.items()]
attrition_table = pd.DataFrame(attrition_rows)
attrition_table.to_csv(ATTRITION_DIR / f"event_attrition_{RUN_LABEL}.csv", index=False)
display(attrition_table)
print(f"Saved attrition table: {ATTRITION_DIR / f'event_attrition_{RUN_LABEL}.csv'}")
print("Note: computed on the current pilot window; rerun after each production window to confirm stage-by-stage behavior at scale.")

In [ ]:
# Linkage-component audit

if "edges_df" not in globals() or "mh_sets_df" not in globals():
    raise RuntimeError("Run the event-labeling/linking cell before this diagnostic.")

LINKAGE_DIR = OUTPUT_ROOT / "diagnostics" / "linkage_audit"
LINKAGE_DIR.mkdir(parents=True, exist_ok=True)

hazards_present = sorted(event_tables)
pair_summary_rows = []
for i in range(len(hazards_present)):
    for j in range(i + 1, len(hazards_present)):
        h1, h2 = hazards_present[i], hazards_present[j]
        n1, n2 = len(event_tables[h1]), len(event_tables[h2])
        candidate_pairs = n1 * n2
        accepted = int(((edges_df["hazard_a"] == h1) & (edges_df["hazard_b"] == h2)).sum()) if not edges_df.empty else 0
        pair_summary_rows.append({
            "hazard_a": h1,
            "hazard_b": h2,
            "n_events_a": n1,
            "n_events_b": n2,
            "candidate_pairs": candidate_pairs,
            "accepted_links": accepted,
            "acceptance_rate": accepted / candidate_pairs if candidate_pairs else np.nan,
        })

linkage_pair_summary = pd.DataFrame(pair_summary_rows)
linkage_pair_summary.to_csv(LINKAGE_DIR / f"linkage_pair_summary_{RUN_LABEL}.csv", index=False)

component_rows = []
for _, mh_row in mh_sets_df.iterrows():
    component_rows.append({
        "mh_set_id": mh_row["mh_set_id"],
        "n_single_events": mh_row["n_single_events"],
        "n_hazard_types": mh_row["n_hazard_types"],
        "hazard_types": mh_row["hazard_types"],
        "duration_days": mh_row["duration_days"],
    })
component_summary = pd.DataFrame(component_rows).sort_values("n_single_events", ascending=False)
component_summary.to_csv(LINKAGE_DIR / f"linkage_component_summary_{RUN_LABEL}.csv", index=False)

largest_components = component_summary.head(10)

print("Multi-hazard component size distribution:")
print(component_summary["n_single_events"].describe().to_string())
print(f"\nLargest components:\n")
display(largest_components)
print(f"\nPairwise candidate vs. accepted links:")
display(linkage_pair_summary)
print(f"\nSaved linkage audit: {LINKAGE_DIR}")
print("Note: sensitivity to MULTIHAZARD_LAG_HOURS (24h/48h/72h) should be re-tested once monthly production runs are available, to check for large transitive chaining.")

## Canonical Production Pipeline

Run this section after Cells 1 and 2. It writes the individual-event, intensity, pair-link, and candidate-group layers for the configured analysis window.

In [ ]:
# Cell 3/5: derive hazard masks for wildfire, terrestrial weather, and space weather

if not DRY_RUN:
    ds_terr = open_daily_stack(terr_files, chunks=CHUNKS)
    ds_space_raw = open_daily_stack(space_files, chunks=CHUNKS)

    # Map 1D space-weather site/gridpt data to 2D analysis grid before thresholding/event extraction.
    ds_target_grid = xr.open_dataset(GRID_PATH)
    ds_space = _grid_space_weather_to_analysis_grid(ds_space_raw, ds_target_grid)

    # Repair sub-daily time axis when multiple daily files share identical intraday stamps.
    # This prevents accidental collapse of a full week into one day during duplicate-time handling.
    if "time" in ds_space.coords and len(space_files) > 1:
        t_sw = pd.DatetimeIndex(pd.to_datetime(ds_space["time"].values, errors="coerce"))
        n_total = len(t_sw)
        n_files = int(len(space_files))

        if n_total > 0 and n_total % n_files == 0:
            n_per_file = n_total // n_files
            unique_days = int(pd.DatetimeIndex(t_sw.normalize()).nunique())
            suspicious_reuse = (n_per_file >= 24) and (unique_days <= max(2, n_files // 2))

            if suspicious_reuse:
                repaired_blocks = []
                base_offsets = None

                # Build reference intraday offsets from first valid block.
                for bi in range(n_files):
                    b0 = bi * n_per_file
                    b1 = (bi + 1) * n_per_file
                    block = pd.DatetimeIndex(t_sw[b0:b1])
                    valid = ~pd.isna(block)
                    if valid.any():
                        base_offsets = block[valid] - block[valid].normalize()
                        if len(base_offsets) == n_per_file:
                            break

                if base_offsets is None or len(base_offsets) != n_per_file:
                    dt_seconds = 60.0
                    if n_per_file >= 2:
                        blk = pd.DatetimeIndex(t_sw[:n_per_file])
                        diffs = np.diff(blk.values.astype("datetime64[ns]")) / np.timedelta64(1, "s")
                        diffs = np.asarray(diffs, dtype=float)
                        diffs = diffs[np.isfinite(diffs) & (diffs > 0)]
                        if diffs.size:
                            dt_seconds = float(np.median(diffs))
                    base_offsets = pd.to_timedelta(np.arange(n_per_file) * dt_seconds, unit="s")

                for i_file, file_day in enumerate(pd.to_datetime(space_files["time"].values)):
                    day0 = pd.Timestamp(file_day).normalize()
                    b0 = i_file * n_per_file
                    b1 = (i_file + 1) * n_per_file
                    block = pd.DatetimeIndex(t_sw[b0:b1])
                    valid = ~pd.isna(block)

                    if valid.any() and int(valid.sum()) == n_per_file:
                        offsets = block - block.normalize()
                    else:
                        offsets = base_offsets

                    repaired_blocks.append(day0 + offsets)

                repaired = pd.DatetimeIndex(np.concatenate([np.asarray(b) for b in repaired_blocks]))
                ds_space = ds_space.assign_coords(time=repaired)
                print(
                    "Repaired space-weather timestamps:",
                    f"{repaired.min()} to {repaired.max()} across {repaired.normalize().nunique()} days",
                )

    # ------------------------------------------------------------------
    # Wildfire source: prefer MYRIAD-style event polygons from GeoPackage.
    # Fallback to legacy daily wildfire grid stack if GeoPackages are unavailable.
    # ------------------------------------------------------------------
    wildfire_source = "unknown"
    wildfire_var = None
    wildfire_events_for_pipeline = None
    wild_da = None

    wildfire_events_gpkg = ROOT / "wildfire" / "fire_events.gpkg"
    wildfire_firms_gpkg = ROOT / "wildfire" / "firms_event_matches.gpkg"

    common_time = pd.date_range(TIME_START, TIME_END, freq="D")
    ny, nx = int(ds_target_grid.sizes["y"]), int(ds_target_grid.sizes["x"])

    use_gpkg_events = bool(gpd is not None and wildfire_events_gpkg.exists())

    if use_gpkg_events:
        events_gdf = gpd.read_file(wildfire_events_gpkg, layer="events")
        if events_gdf.crs is None:
            events_gdf = events_gdf.set_crs("EPSG:4326")
        else:
            events_gdf = events_gdf.to_crs("EPSG:4326")

        events_gdf["start_date"] = pd.to_datetime(events_gdf["start_date"], errors="coerce")
        events_gdf["end_date"] = pd.to_datetime(events_gdf["end_date"], errors="coerce")
        events_gdf["final_area_km2"] = pd.to_numeric(events_gdf["final_area_km2"], errors="coerce")

        # Match MYRIAD wildfire filter: keep events with final polygon size >= 5 km2.
        events_gdf = events_gdf[events_gdf["final_area_km2"] >= float(MIN_WILDFIRE_EVENT_AREA_KM2)].copy()

        tune_start = pd.Timestamp(TIME_START)
        tune_end = pd.Timestamp(TIME_END)
        events_gdf = events_gdf[
            (events_gdf["end_date"] >= tune_start)
            & (events_gdf["start_date"] <= tune_end)
            & events_gdf["start_date"].notna()
            & events_gdf["end_date"].notna()
        ].copy()

        # Optional FRP augmentation keyed by event_id.
        if wildfire_firms_gpkg.exists():
            firms_gdf = gpd.read_file(wildfire_firms_gpkg, layer="firms_matches")
            firms_gdf["frp"] = pd.to_numeric(firms_gdf["frp"], errors="coerce")
            frp_summary = firms_gdf.groupby("event_id", dropna=True).agg(
                n_firms_pts=("frp", "size"),
                frp_sum=("frp", "sum"),
                frp_mean=("frp", "mean"),
                frp_max=("frp", "max"),
            ).reset_index()
            events_gdf = events_gdf.merge(frp_summary, on="event_id", how="left")
        else:
            events_gdf["n_firms_pts"] = np.nan
            events_gdf["frp_sum"] = np.nan
            events_gdf["frp_mean"] = np.nan
            events_gdf["frp_max"] = np.nan

        wildfire_events_for_pipeline = events_gdf.drop(columns=["geometry"]).copy()

        flat_lat = ds_target_grid["lat"].values.ravel()
        flat_lon = ds_target_grid["lon"].values.ravel()
        grid_points = gpd.GeoDataFrame(
            {
                "flat_idx": np.arange(ny * nx, dtype=np.int64),
                "lat": flat_lat,
                "lon": flat_lon,
            },
            geometry=gpd.points_from_xy(flat_lon, flat_lat),
            crs="EPSG:4326",
        )

        # For small polygons relative to 50 km cells, use nearest-cell fallback to avoid false zero occupancy.
        grid_tree = cKDTree(np.column_stack([flat_lat.astype(float), flat_lon.astype(float)]))

        day_to_idx = {pd.Timestamp(d): i for i, d in enumerate(common_time)}
        wildfire_mask_arr = np.zeros((len(common_time), ny, nx), dtype=bool)
        wildfire_value_arr = np.full((len(common_time), ny, nx), np.nan, dtype=np.float32)

        for _, ev in events_gdf.iterrows():
            geom = ev.geometry
            if geom is None or geom.is_empty:
                continue

            inside = grid_points.intersects(geom)
            flat_ids = grid_points.loc[inside, "flat_idx"].values.astype(np.int64)

            centroid = geom.centroid
            if np.isfinite(centroid.y) and np.isfinite(centroid.x):
                _, nearest_flat = grid_tree.query([float(centroid.y), float(centroid.x)], k=1)
                flat_ids = np.unique(np.concatenate([flat_ids, np.asarray([nearest_flat], dtype=np.int64)]))

            if flat_ids.size == 0:
                continue

            ev_start = max(pd.Timestamp(ev["start_date"]).normalize(), tune_start)
            ev_end = min(pd.Timestamp(ev["end_date"]).normalize(), tune_end)
            if ev_end < ev_start:
                continue

            ev_days = pd.date_range(ev_start, ev_end, freq="D")
            time_ids = [day_to_idx[d] for d in ev_days if d in day_to_idx]
            if len(time_ids) == 0:
                continue

            score_val = ev.get("frp_mean", np.nan)
            if not np.isfinite(score_val):
                score_val = ev.get("final_area_km2", np.nan)
            score_val = np.float32(score_val) if np.isfinite(score_val) else np.float32(1.0)

            for ti in time_ids:
                sl = wildfire_mask_arr[ti].reshape(-1)
                sl[flat_ids] = True

                vl = wildfire_value_arr[ti].reshape(-1)
                prev = vl[flat_ids]
                # Keep strongest event signal per cell/day if multiple events overlap.
                vl[flat_ids] = np.where(np.isnan(prev), score_val, np.maximum(prev, score_val))

        wildfire_mask = xr.DataArray(
            wildfire_mask_arr,
            coords={"time": common_time, "y": ds_target_grid["y"].values, "x": ds_target_grid["x"].values},
            dims=("time", "y", "x"),
        ).fillna(False)

        wild_da = xr.DataArray(
            wildfire_value_arr,
            coords={"time": common_time, "y": ds_target_grid["y"].values, "x": ds_target_grid["x"].values},
            dims=("time", "y", "x"),
        )

        wildfire_source = "gpkg_events"

    else:
        ds_wild = open_daily_stack(wildfire_files, chunks=CHUNKS)
        wildfire_var = infer_var(ds_wild, ["fire_mask", "burned_area", "fire_area_km2", "active_fire", "frp"])

        # Legacy wildfire mask from daily gridded product.
        wild_da = _prepare_hazard_time(ds_wild[wildfire_var], "wildfire")
        if "frp" in wildfire_var.lower():
            frp_pos = wild_da.where(wild_da > 0)
            frp_thr = _quantile_threshold(frp_pos, THRESHOLDS["wildfire_frp_pct"])
            wildfire_mask = ((wild_da > 0) & (wild_da > frp_thr)).fillna(False)
        else:
            wildfire_mask = (wild_da > 0).fillna(False)
            if any(k in wildfire_var.lower() for k in ["area", "km2"]):
                wildfire_mask = (wild_da >= MIN_WILDFIRE_EVENT_AREA_KM2).fillna(False)

        wildfire_source = "daily_grid_fallback"

    temp_var = infer_var(ds_terr, ["T2M", "TMP2m", "temperature", "air_temperature", "tmean"], required=False)

    # Wind priority: use i10fg (10 m wind gust) first to align with MYRIAD-style gust hazard definitions.
    wind_var = infer_var(
        ds_terr,
        ["i10fg", "I10FG", "wind_gust", "gust", "MAX_SHEAR", "VII_00.50", "wind"],
        required=False,
    )

    precip_var = infer_var(ds_terr, ["MultiSensor_QPE_24H_Pass2_00.00", "qpe", "precip", "rain"], required=False)
    hail_var = infer_var(ds_terr, ["MESH_00.50", "MAXSIZE", "hail"], required=False)
    lightning_var = infer_var(ds_terr, ["FLASH_QPE_ARIMAX_00.00", "flash", "lightning"], required=False)

    ex_var = infer_var(ds_space, ["Ex"], required=True)
    ey_var = infer_var(ds_space, ["Ey"], required=True)

    hazard_masks: Dict[str, xr.DataArray] = {"wildfire": wildfire_mask}

    # Heatwave / coldwave
    if temp_var is not None:
        temp_da = _prepare_hazard_time(ds_terr[temp_var], "heatwave")
        t_hi = _quantile_threshold(temp_da, THRESHOLDS["heatwave_pct"])
        t_lo = _quantile_threshold(temp_da, THRESHOLDS["coldwave_pct"])

        heatwave_raw = temp_da > t_hi
        coldwave_raw = temp_da < t_lo

        heatwave = _rolling_consecutive(heatwave_raw, MIN_CONSECUTIVE_DAYS["heatwave"])
        coldwave = _rolling_consecutive(coldwave_raw, MIN_CONSECUTIVE_DAYS["coldwave"])

        heatwave = heatwave & (temp_da.max(dim="time", skipna=True) > 0)
        coldwave = coldwave & (temp_da.min(dim="time", skipna=True) < 0)

        hazard_masks["heatwave"] = heatwave.fillna(False)
        hazard_masks["coldwave"] = coldwave.fillna(False)
    else:
        warnings.warn("Temperature variable not found in terrestrial weather dataset; skipping heatwave/coldwave.")

    # Extreme wind
    if wind_var is not None:
        wind_da = _prepare_hazard_time(ds_terr[wind_var], "extreme_wind")
        if TEMPORAL_MODE.get("extreme_wind", "daily") == "daily":
            wind_da = _safe_daily_time_coord(wind_da)

        wind_thr = _quantile_threshold(wind_da, THRESHOLDS["wind_pct"])
        wind_mask = (wind_da > wind_thr).fillna(False)

        if TEMPORAL_MODE.get("extreme_wind", "daily") == "subdaily":
            wind_mask = _rolling_consecutive_steps(
                wind_mask,
                MIN_CONSECUTIVE_STEPS.get("extreme_wind", 1),
            )

        hazard_masks["extreme_wind"] = wind_mask
    else:
        warnings.warn("Wind-like variable not found; skipping extreme_wind.")

    # Extreme precipitation / drought proxy
    if precip_var is not None:
        p_da = _prepare_hazard_time(ds_terr[precip_var], "extreme_precip")
        p_hi = _quantile_threshold(p_da, THRESHOLDS["precip_high_pct"])
        p_lo = _quantile_threshold(p_da, THRESHOLDS["precip_low_pct"])

        hazard_masks["extreme_precip"] = (p_da > p_hi).fillna(False)

        low_p = (p_da < p_lo).fillna(False)
        drought_proxy = _rolling_consecutive(low_p, MIN_CONSECUTIVE_DAYS["drought_proxy"])
        hazard_masks["drought_proxy"] = drought_proxy.fillna(False)
    else:
        warnings.warn("Precipitation variable not found; skipping extreme_precip and drought_proxy.")

    # Hail / lightning
    if hail_var is not None:
        hail_da = _prepare_hazard_time(ds_terr[hail_var], "hail")
        hail_thr = _quantile_threshold(hail_da, THRESHOLDS["hail_pct"])
        hazard_masks["hail"] = (hail_da > hail_thr).fillna(False)
    else:
        warnings.warn("Hail-like variable not found; skipping hail.")

    if lightning_var is not None:
        li_da = _prepare_hazard_time(ds_terr[lightning_var], "lightning")
        li_thr = _quantile_threshold(li_da, THRESHOLDS["lightning_pct"])
        hazard_masks["lightning"] = (li_da > li_thr).fillna(False)
    else:
        warnings.warn("Lightning-like variable not found; skipping lightning.")

    # Space weather from Ex/Ey on analysis grid, using the validated operational definition:
    # 5-min max aggregation, per-cell 95th percentile threshold, 5-step persistence.
    e_mag = np.hypot(ds_space[ex_var], ds_space[ey_var])
    space_source = _prepare_hazard_time(e_mag, "space_weather_extreme")

    sw_agg_minutes = int(SPACE_WEATHER_OPERATIONAL["aggregation_minutes"])
    if sw_agg_minutes > 1:
        space_source = space_source.resample(time=f"{sw_agg_minutes}min").max(skipna=True)
        space_source = _prepare_hazard_time(space_source, "space_weather_extreme")

    space_threshold = _quantile_threshold(space_source, SPACE_WEATHER_OPERATIONAL["percentile"])
    sw_mask = (space_source > space_threshold).fillna(False)
    sw_mask = _rolling_consecutive_steps(sw_mask, SPACE_WEATHER_OPERATIONAL["persistence_steps"])

    hazard_masks["space_weather_extreme"] = sw_mask
    e_mag = space_source  # downstream diagnostics reference the operational (aggregated) field

    # Harmonize each hazard to its own mode output timeline.
    for h in list(hazard_masks):
        da = hazard_masks[h]
        if TEMPORAL_MODE.get(h, "daily") == "daily":
            da = _safe_daily_time_coord(da)
            hazard_masks[h] = da.reindex(time=common_time, fill_value=False)
        else:
            hazard_masks[h] = _safe_subdaily_time_coord(da)

    print("Constructed hazard masks:")
    print("  Selected variables:")
    print(f"    wildfire_source={wildfire_source}")
    if wildfire_var is not None:
        print(f"    wildfire_var={wildfire_var}")
    print(f"    wind_var={wind_var}")
    print(f"    precip_var={precip_var}")
    if wildfire_events_for_pipeline is not None:
        print(f"    wildfire_events_used={len(wildfire_events_for_pipeline)} (area >= {MIN_WILDFIRE_EVENT_AREA_KM2} km2)")
    print(f"    space_weather_operational={SPACE_WEATHER_OPERATIONAL}")
    for h, da in hazard_masks.items():
        frac = float(da.mean().compute() if hasattr(da.mean(), "compute") else da.mean())
        print(f"  {h:24s} mode={TEMPORAL_MODE.get(h, 'daily'):8s} active-cell fraction={frac:.5f}")
else:
    hazard_masks = {}
    print("DRY_RUN=True, skipped hazard-mask computation")

In [ ]:
# Correct hail/lightning sources and expose convective-storm proxy fields

if "ds_terr" not in globals() or "hazard_masks" not in globals():
    raise RuntimeError("Run the hazard-mask construction cell before this source correction.")

hail_var = infer_var(ds_terr, ["MAXSIZE", "MESH_00.50", "hail"], required=False)
lightning_var = infer_var(ds_terr, ["NLDN_CG_001min_AvgDensity_00.00", "NLDN_CG_015min_AvgDensity_00.00", "flash", "lightning"], required=False)

if hail_var is None:
    warnings.warn("No MAXSIZE or MESH hail variable found; hail unavailable.")
else:
    hail_candidate = _prepare_hazard_time(ds_terr[hail_var], "hail")
    hail_values = np.asarray(hail_candidate.values, dtype=float)
    if np.isfinite(hail_values).any() and np.nanmax(hail_values) > 0:
        hail_da = hail_candidate
        hail_thr = _quantile_threshold(hail_da, THRESHOLDS["hail_pct"])
        hazard_masks["hail"] = (hail_da > hail_thr).fillna(False)
    else:
        hail_var = None
        hazard_masks.pop("hail", None)
        warnings.warn("Hail source has no positive valid values; hail unavailable for this window.")

if lightning_var is None:
    warnings.warn("No NLDN lightning-density variable found; lightning unavailable.")
else:
    lightning_candidate = _prepare_hazard_time(ds_terr[lightning_var], "lightning")
    lightning_values = np.asarray(lightning_candidate.values, dtype=float)
    if np.isfinite(lightning_values).any() and np.nanmax(lightning_values) > 0:
        li_da = lightning_candidate
        li_thr = _quantile_threshold(li_da, THRESHOLDS["lightning_pct"])
        hazard_masks["lightning"] = (li_da > li_thr).fillna(False)
    else:
        lightning_var = None
        hazard_masks.pop("lightning", None)
        warnings.warn("Lightning source has no positive valid values; lightning unavailable for this window.")

convective_proxy_fields = {}
for proxy_name, proxy_spec in CONVECTIVE_PROXY_VARIABLES.items():
    proxy_var = proxy_spec["variable"]
    if proxy_var in ds_terr.data_vars:
        convective_proxy_fields[proxy_name] = _prepare_hazard_time(ds_terr[proxy_var], "lightning")

print(f"Corrected hail source: {hail_var or 'unavailable'}")
print(f"Corrected lightning source: {lightning_var or 'unavailable'}")
print("Available convective proxy fields:", ", ".join(convective_proxy_fields) or "none")
for proxy_name, proxy_da in convective_proxy_fields.items():
    print(f"  {proxy_name}: {proxy_da.dims}, {proxy_da.sizes.get('time', 0)} time samples")

In [ ]:
# Inspect corrected hail and lightning source conventions

source_inspection_rows = []
for variable_name in ["MAXSIZE", "MESH_00.50", "NLDN_CG_001min_AvgDensity_00.00", "NLDN_CG_015min_AvgDensity_00.00"]:
    if variable_name not in ds_terr.data_vars:
        continue
    source_da = ds_terr[variable_name]
    source_values = np.asarray(source_da.values, dtype=float)
    finite_values = source_values[np.isfinite(source_values)]
    source_inspection_rows.append({
        "variable": variable_name,
        "dims": str(source_da.dims),
        "shape": str(source_da.shape),
        "units": source_da.attrs.get("units", "missing"),
        "long_name": source_da.attrs.get("long_name", source_da.attrs.get("description", "missing")),
        "fill_value": source_da.encoding.get("_FillValue", source_da.attrs.get("_FillValue", "missing")),
        "minimum": float(np.min(finite_values)) if finite_values.size else np.nan,
        "p01": float(np.percentile(finite_values, 1)) if finite_values.size else np.nan,
        "median": float(np.median(finite_values)) if finite_values.size else np.nan,
        "p99": float(np.percentile(finite_values, 99)) if finite_values.size else np.nan,
        "maximum": float(np.max(finite_values)) if finite_values.size else np.nan,
        "negative_fraction": float((finite_values < 0).mean()) if finite_values.size else np.nan,
        "positive_fraction": float((finite_values > 0).mean()) if finite_values.size else np.nan,
    })

source_inspection = pd.DataFrame(source_inspection_rows)
source_inspection.to_csv(OUTPUT_DIR / "hail_lightning_source_inspection.csv", index=False)
print("Corrected hail/lightning source inspection:")
display(source_inspection)
for variable_name in source_inspection["variable"]:
    print(variable_name, dict(ds_terr[variable_name].attrs), dict(ds_terr[variable_name].encoding))

In [ ]:
# Full terrestrial inventory check for hail and lightning fields

if "TERRESTRIAL_DIR" not in globals():
    raise RuntimeError("Run the configuration cell before this inventory check.")

inventory_rows = []
for path in sorted(TERRESTRIAL_DIR.glob("weather_CONUS_*.nc")):
    file_date = _extract_date_from_name(path.name)
    if file_date is None:
        continue
    try:
        with xr.open_dataset(path, decode_times=False) as dataset:
            data_vars = set(dataset.data_vars)
            present = {
                variable_name: variable_name in data_vars
                for variable_name in [
                    "MAXSIZE",
                    "MESH_00.50",
                    "NLDN_CG_001min_AvgDensity_00.00",
                    "NLDN_CG_015min_AvgDensity_00.00",
                    "EchoTop_18_00.50",
                    "EchoTop_50_00.50",
                ]
            }
            inventory_rows.append({"date": file_date, "path": str(path), **present})
    except Exception as error:
        inventory_rows.append({"date": file_date, "path": str(path), "read_error": str(error)})

terrestrial_variable_inventory = pd.DataFrame(inventory_rows).sort_values("date").reset_index(drop=True)
variable_columns = [column for column in terrestrial_variable_inventory.columns if column not in {"date", "path", "read_error"}]
presence_summary = terrestrial_variable_inventory[variable_columns].sum().rename("files_with_variable").to_frame()
presence_summary["total_files"] = len(terrestrial_variable_inventory)
presence_summary["file_presence_rate"] = presence_summary["files_with_variable"] / presence_summary["total_files"]

representative_rows = []
for _, inventory_row in terrestrial_variable_inventory.iloc[
    sorted(set([0, len(terrestrial_variable_inventory) // 2, len(terrestrial_variable_inventory) - 1]))
].iterrows():
    path = Path(inventory_row["path"])
    with xr.open_dataset(path, decode_times=False) as dataset:
        for variable_name in variable_columns:
            if not inventory_row.get(variable_name, False):
                continue
            source_values = np.asarray(dataset[variable_name].values, dtype=float)
            finite_values = source_values[np.isfinite(source_values)]
            representative_rows.append({
                "date": inventory_row["date"],
                "variable": variable_name,
                "dims": str(dataset[variable_name].dims),
                "shape": str(dataset[variable_name].shape),
                "minimum": float(np.min(finite_values)) if finite_values.size else np.nan,
                "maximum": float(np.max(finite_values)) if finite_values.size else np.nan,
                "positive_fraction": float((finite_values > 0).mean()) if finite_values.size else np.nan,
                "negative_fraction": float((finite_values < 0).mean()) if finite_values.size else np.nan,
                "attributes": dict(dataset[variable_name].attrs),
            })

representative_variable_values = pd.DataFrame(representative_rows)
field_inventory_dir = OUTPUT_ROOT / "diagnostics" / "source_field_inventory"
field_inventory_dir.mkdir(parents=True, exist_ok=True)
terrestrial_variable_inventory.to_csv(field_inventory_dir / "terrestrial_hail_lightning_variable_presence.csv", index=False)
presence_summary.to_csv(field_inventory_dir / "terrestrial_hail_lightning_presence_summary.csv")
representative_variable_values.to_csv(field_inventory_dir / "terrestrial_hail_lightning_representative_values.csv", index=False)

print("Full terrestrial file inventory:")
display(presence_summary)
print("Representative field dimensions and values:")
display(representative_variable_values.drop(columns=["attributes"]))

In [ ]:
# Use gauge-corrected one-hour accumulation for sub-daily extreme precipitation

if "hazard_masks" not in globals() or "ds_terr" not in globals():
    raise RuntimeError("Run the hazard-mask construction cell before this override.")

short_term_precip_var = HAZARD_RULES["extreme_precip"]["source_variable"]
if short_term_precip_var not in ds_terr.data_vars:
    raise KeyError(f"Required short-term precipitation variable is missing: {short_term_precip_var}")

precip_var = short_term_precip_var
p_da = _prepare_hazard_time(ds_terr[precip_var], "extreme_precip")
p_hi = _quantile_threshold(p_da, THRESHOLDS["precip_high_pct"])
hazard_masks["extreme_precip"] = (p_da > p_hi).fillna(False)
hazard_masks["extreme_precip"] = _safe_subdaily_time_coord(hazard_masks["extreme_precip"])

active_fraction = float(hazard_masks["extreme_precip"].mean().compute() if hasattr(hazard_masks["extreme_precip"].mean(), "compute") else hazard_masks["extreme_precip"].mean())
print(f"Extreme precipitation source: {precip_var}")
print(f"Extreme precipitation active fraction: {active_fraction:.5f}")

In [ ]:
# Add sub-daily extreme-temperature masks

if "hazard_masks" not in globals() or "ds_terr" not in globals() or temp_var is None:
    raise RuntimeError("Run the hazard-mask construction cell before this cell.")

# Heatwave/coldwave remain daily MYRIAD hazards. These separate hazards retain the
# hourly temperature field and compare each hour with its grid-cell-specific monthly
# distribution, which avoids treating ordinary summer and winter temperatures alike.
temp_hourly = _safe_subdaily_time_coord(ds_terr[temp_var])
if getattr(temp_hourly, "chunks", None) is not None:
    temp_hourly = temp_hourly.chunk({"time": -1})
monthly_temp = temp_hourly.groupby("time.month")

extreme_heat_threshold_month = monthly_temp.quantile(THRESHOLDS["extreme_heat_pct"] / 100.0, dim="time", skipna=True)
extreme_cold_threshold_month = monthly_temp.quantile(THRESHOLDS["extreme_cold_pct"] / 100.0, dim="time", skipna=True)
if "quantile" in extreme_heat_threshold_month.dims:
    extreme_heat_threshold_month = extreme_heat_threshold_month.squeeze("quantile", drop=True)
if "quantile" in extreme_cold_threshold_month.dims:
    extreme_cold_threshold_month = extreme_cold_threshold_month.squeeze("quantile", drop=True)

sample_months = temp_hourly["time"].dt.month
extreme_heat_threshold = extreme_heat_threshold_month.sel(month=sample_months)
extreme_cold_threshold = extreme_cold_threshold_month.sel(month=sample_months)

extreme_heat_raw = (temp_hourly > extreme_heat_threshold).fillna(False)
extreme_cold_raw = (temp_hourly < extreme_cold_threshold).fillna(False)
hazard_masks["extreme_heat"] = _rolling_consecutive_steps(extreme_heat_raw, MIN_CONSECUTIVE_STEPS["extreme_heat"])
hazard_masks["extreme_cold"] = _rolling_consecutive_steps(extreme_cold_raw, MIN_CONSECUTIVE_STEPS["extreme_cold"])

for hazard in ("extreme_heat", "extreme_cold"):
    hazard_masks[hazard] = _safe_subdaily_time_coord(hazard_masks[hazard])
    active_fraction = float(hazard_masks[hazard].mean().compute() if hasattr(hazard_masks[hazard].mean(), "compute") else hazard_masks[hazard].mean())
    print(f"{hazard}: monthly local threshold, {MIN_CONSECUTIVE_STEPS[hazard]}-hour persistence, active fraction={active_fraction:.5f}")

In [ ]:
# Cell 4/5: identify single-hazard events and compile MYRIAD-style multi-hazard event sets


def _event_connectivity_structure(allow_spatial_drift: bool = True) -> np.ndarray:
    """Connectivity kernel for [time, y, x] labeling.

    Includes 4-neighbor spatial links at the same timestep, and temporal links to
    adjacent timesteps. With drift enabled, temporal links also include 4-neighbor
    cells so moving footprints remain one event.
    """
    s = np.zeros((3, 3, 3), dtype=bool)

    # Same-time 4-neighborhood + center.
    s[1, 1, 1] = True
    s[1, 0, 1] = True
    s[1, 2, 1] = True
    s[1, 1, 0] = True
    s[1, 1, 2] = True

    # Adjacent-time same-cell links.
    s[0, 1, 1] = True
    s[2, 1, 1] = True

    if allow_spatial_drift:
        for t in (0, 2):
            s[t, 0, 1] = True
            s[t, 2, 1] = True
            s[t, 1, 0] = True
            s[t, 1, 2] = True

    return s


def _bridge_short_time_gaps(arr: np.ndarray, max_gap_steps: int) -> np.ndarray:
    """Fill short below-threshold dips along time so events do not split too easily."""
    if max_gap_steps <= 0:
        return arr
    kernel = np.ones((int(max_gap_steps) + 1, 1, 1), dtype=bool)
    return ndimage.binary_closing(arr.astype(bool), structure=kernel)


def label_single_hazard_events(mask_da: xr.DataArray, hazard_name: str) -> Tuple[pd.DataFrame, np.ndarray]:
    """
    Label connected 3D components (time, y, x) as single-hazard events.
    """
    arr = _safe_to_numpy(mask_da)
    if arr.ndim != 3:
        raise ValueError(f"Expected [time, y, x] for {hazard_name}, got shape {arr.shape}")

    gap_steps = int(EVENT_END_GAP_STEPS.get(hazard_name, 0))
    arr = _bridge_short_time_gaps(arr, gap_steps)

    structure = _event_connectivity_structure(bool(ALLOW_SPATIAL_DRIFT_LINKING))
    labels, n_labels = ndimage.label(arr, structure=structure)

    times = pd.to_datetime(mask_da["time"].values)

    if len(times) >= 2:
        dt_hours = float(np.nanmedian(np.diff(times) / np.timedelta64(1, "h")))
        if not np.isfinite(dt_hours) or dt_hours <= 0:
            dt_hours = 24.0
    else:
        dt_hours = 24.0

    min_cells = int(MIN_EVENT_FOOTPRINT_CELLS.get(hazard_name, 1))
    min_active_steps = int(MIN_EVENT_ACTIVE_CELL_STEPS.get(hazard_name, 1))

    if n_labels == 0:
        return pd.DataFrame(columns=[
            "event_id", "hazard", "start_time", "end_time", "duration_hours", "duration_days",
            "time_step_hours", "active_cell_steps", "footprint_cells", "footprint_km2"
        ]), labels

    ti, yi, xi = np.where(labels > 0)
    if len(ti) == 0:
        return pd.DataFrame(columns=[
            "event_id", "hazard", "start_time", "end_time", "duration_hours", "duration_days",
            "time_step_hours", "active_cell_steps", "footprint_cells", "footprint_km2"
        ]), labels

    lbl = labels[ti, yi, xi].astype(np.int64)
    ny, nx = labels.shape[1], labels.shape[2]
    ncells = int(ny * nx)

    active_cell_steps_counts = np.bincount(lbl, minlength=n_labels + 1)

    tmin = np.full(n_labels + 1, np.iinfo(np.int64).max, dtype=np.int64)
    tmax = np.full(n_labels + 1, -1, dtype=np.int64)
    ti64 = ti.astype(np.int64)
    np.minimum.at(tmin, lbl, ti64)
    np.maximum.at(tmax, lbl, ti64)

    flat_cell = yi.astype(np.int64) * nx + xi.astype(np.int64)
    keys = lbl * ncells + flat_cell
    ukeys = np.unique(keys)
    u_lbl = (ukeys // ncells).astype(np.int64)
    footprint_counts = np.bincount(u_lbl, minlength=n_labels + 1)

    records: List[dict] = []
    for event_idx in range(1, n_labels + 1):
        active_cell_steps = int(active_cell_steps_counts[event_idx])
        if active_cell_steps == 0:
            continue
        if active_cell_steps < min_active_steps:
            continue

        footprint_cells = int(footprint_counts[event_idx])
        if footprint_cells < min_cells:
            continue

        start_i = int(tmin[event_idx])
        end_i = int(tmax[event_idx])
        if end_i < start_i:
            continue

        start_t = pd.Timestamp(times[start_i])
        end_t = pd.Timestamp(times[end_i])
        duration_hours = float((end_t - start_t) / pd.Timedelta(hours=1) + dt_hours)
        duration_days = duration_hours / 24.0

        if TEMPORAL_MODE.get(hazard_name, "daily") == "daily" and duration_days < float(MIN_EVENT_DURATION_DAYS):
            continue

        footprint_km2 = footprint_cells * 2500.0

        records.append(
            {
                "event_id": f"{hazard_name}_{event_idx:06d}",
                "hazard": hazard_name,
                "start_time": start_t,
                "end_time": end_t,
                "duration_hours": duration_hours,
                "duration_days": duration_days,
                "time_step_hours": dt_hours,
                "active_cell_steps": active_cell_steps,
                "footprint_cells": footprint_cells,
                "footprint_km2": footprint_km2,
            }
        )

    return pd.DataFrame(records), labels


def _event_voxel_index(labels: np.ndarray, event_numeric_id: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    return np.where(labels == event_numeric_id)


def _build_event_cache(tbl: pd.DataFrame, labels: np.ndarray) -> Dict[str, dict]:
    """Cache per-event time bounds and footprint set once to avoid repeated np.where in pair loops."""
    cache: Dict[str, dict] = {}
    for _, row in tbl.iterrows():
        eid = str(row["event_id"])
        enum = int(eid.split("_")[-1])
        _, yy, xx = _event_voxel_index(labels, enum)
        if len(yy) == 0:
            continue

        cells = set(zip(yy.tolist(), xx.tolist()))
        y_min, y_max = int(yy.min()), int(yy.max())
        x_min, x_max = int(xx.min()), int(xx.max())

        cache[eid] = {
            "start": pd.Timestamp(row["start_time"]),
            "end": pd.Timestamp(row["end_time"]),
            "cells": cells,
            "y_min": y_min,
            "y_max": y_max,
            "x_min": x_min,
            "x_max": x_max,
        }
    return cache


def _events_are_linked_cached(a: dict, b: dict, hazard_a: str, hazard_b: str) -> bool:
    # Temporal criterion: identify which event leads (if either) and apply that pair's lag.
    if a["start"] <= b["end"] and b["start"] <= a["end"]:
        gap_hours = 0.0
        lag_hours = _lag_hours_for_ordered_pair(hazard_a, hazard_b)
    elif b["start"] > a["end"]:
        gap_hours = float((b["start"] - a["end"]) / pd.Timedelta(hours=1))
        lag_hours = _lag_hours_for_ordered_pair(hazard_a, hazard_b)
    else:
        gap_hours = float((a["start"] - b["end"]) / pd.Timedelta(hours=1))
        lag_hours = _lag_hours_for_ordered_pair(hazard_b, hazard_a)

    if gap_hours > lag_hours:
        return False

    # Fast bbox adjacency prefilter with one-cell halo.
    if a["y_max"] < (b["y_min"] - 1) or b["y_max"] < (a["y_min"] - 1):
        return False
    if a["x_max"] < (b["x_min"] - 1) or b["x_max"] < (a["x_min"] - 1):
        return False

    a_cells = a["cells"]
    b_cells = b["cells"]

    if a_cells & b_cells:
        return True

    # Iterate smaller footprint for adjacency checks.
    src, tgt = (a_cells, b_cells) if len(a_cells) <= len(b_cells) else (b_cells, a_cells)
    for y, x in src:
        if (y - 1, x) in tgt or (y + 1, x) in tgt or (y, x - 1) in tgt or (y, x + 1) in tgt:
            return True
    return False


if not DRY_RUN:
    import networkx as nx_local

    event_tables: Dict[str, pd.DataFrame] = {}
    label_cubes: Dict[str, np.ndarray] = {}

    for hz, da in hazard_masks.items():
        print(f"Labeling single-hazard events: {hz}")
        tbl, lbl = label_single_hazard_events(da, hz)

        # Wildfire GeoPackage events are already filtered by final_area_km2 in Cell 3.
        # Avoid applying a second footprint-based filter derived from coarse grid occupancy.
        if hz == "wildfire" and str(globals().get("wildfire_source", "")).lower() != "gpkg_events":
            tbl = tbl[tbl["footprint_km2"] >= MIN_WILDFIRE_EVENT_AREA_KM2].copy()

        event_tables[hz] = tbl
        label_cubes[hz] = lbl
        tbl.to_csv(OUTPUT_DIR / f"single_hazard_events_{hz}.csv", index=False)

    single_hazard_events = pd.concat(event_tables.values(), ignore_index=True)
    single_hazard_events = single_hazard_events.sort_values(["start_time", "hazard"]).reset_index(drop=True)
    single_hazard_events.to_csv(OUTPUT_DIR / "single_hazard_events_all.csv", index=False)

    print(f"Total single-hazard events: {len(single_hazard_events)}")
    print("Per-hazard counts:")
    print(single_hazard_events["hazard"].value_counts().to_string())

    G = nx_local.Graph()
    for _, row in single_hazard_events.iterrows():
        G.add_node(row["event_id"], hazard=row["hazard"])

    hazards = sorted(event_tables)
    edge_rows = []

    # Build caches once per hazard to avoid repeated expensive voxel extraction.
    event_cache_by_hazard: Dict[str, Dict[str, dict]] = {
        hz: _build_event_cache(event_tables[hz], label_cubes[hz]) for hz in hazards
    }

    for i in range(len(hazards)):
        for j in range(i + 1, len(hazards)):
            h1, h2 = hazards[i], hazards[j]
            c1 = event_cache_by_hazard[h1]
            c2 = event_cache_by_hazard[h2]
            if not c1 or not c2:
                continue

            print(f"Linking pairs: {h1} vs {h2} ({len(c1)} x {len(c2)})")

            for eid1, ev1 in c1.items():
                for eid2, ev2 in c2.items():
                    if _events_are_linked_cached(ev1, ev2, h1, h2):
                        G.add_edge(eid1, eid2)
                        edge_rows.append(
                            {
                                "event_id_a": eid1,
                                "hazard_a": h1,
                                "event_id_b": eid2,
                                "hazard_b": h2,
                                "lag_hours": _lag_hours_for_ordered_pair(h1, h2),
                            }
                        )

    edges_df = pd.DataFrame(edge_rows)
    edges_df.to_csv(OUTPUT_DIR / "multi_hazard_event_links.csv", index=False)

    components = list(nx_local.connected_components(G))
    set_rows = []
    membership_rows = []

    for k, comp in enumerate(components, start=1):
        comp_events = single_hazard_events[single_hazard_events["event_id"].isin(comp)].copy()
        if comp_events.empty:
            continue
        hazards_in_set = sorted(comp_events["hazard"].unique().tolist())
        start = comp_events["start_time"].min()
        end = comp_events["end_time"].max()

        set_id = f"mh_set_{k:06d}"
        set_rows.append(
            {
                "mh_set_id": set_id,
                "n_single_events": int(len(comp_events)),
                "n_hazard_types": int(len(hazards_in_set)),
                "hazard_types": "|".join(hazards_in_set),
                "start_time": start,
                "end_time": end,
                "duration_days": float((pd.Timestamp(end) - pd.Timestamp(start)) / pd.Timedelta(days=1) + 1.0),
            }
        )

        for eid in comp:
            membership_rows.append({"mh_set_id": set_id, "event_id": eid})

    mh_sets_df = pd.DataFrame(set_rows).sort_values(["start_time", "mh_set_id"])
    mh_members_df = pd.DataFrame(membership_rows)

    mh_sets_df.to_csv(OUTPUT_DIR / "multi_hazard_event_sets.csv", index=False)
    mh_members_df.to_csv(OUTPUT_DIR / "multi_hazard_event_membership.csv", index=False)

    print(f"Identified multi-hazard event sets: {len(mh_sets_df)}")

    if gpd is not None and Point is not None:
        print("Creating optional centroid geopackage...")
        centroid_rows = []
        template = next(iter(hazard_masks.values()))
        lat_name = next((c for c in ["lat", "latitude", "y"] if c in template.coords), None)
        lon_name = next((c for c in ["lon", "longitude", "x"] if c in template.coords), None)

        if lat_name is not None and lon_name is not None:
            lat_vals = template[lat_name].values
            lon_vals = template[lon_name].values

            for _, mrow in mh_members_df.iterrows():
                eid = mrow["event_id"]
                hz = eid.rsplit("_", 1)[0]
                enum = int(eid.split("_")[-1])
                lbl = label_cubes[hz]
                _, yy, xx = np.where(lbl == enum)
                if len(yy) == 0:
                    continue
                y0 = int(np.median(yy))
                x0 = int(np.median(xx))

                if np.ndim(lat_vals) == 1 and np.ndim(lon_vals) == 1:
                    lat = float(lat_vals[y0])
                    lon = float(lon_vals[x0])
                else:
                    lat = float(lat_vals[y0, x0])
                    lon = float(lon_vals[y0, x0])

                centroid_rows.append(
                    {
                        "mh_set_id": mrow["mh_set_id"],
                        "event_id": eid,
                        "hazard": hz,
                        "geometry": Point(lon, lat),
                    }
                )

            if centroid_rows:
                gdf = gpd.GeoDataFrame(centroid_rows, geometry="geometry", crs="EPSG:4326")
                gdf.to_file(OUTPUT_DIR / "multi_hazard_event_centroids.gpkg", driver="GPKG")

    print("Pipeline complete. Outputs written to:", OUTPUT_DIR)
else:
    print("DRY_RUN=True, skipped event labeling and multi-hazard linking")

## Optional Development Diagnostics: Event and Group Validation

Use the following cells to inspect intensities, spatial overlap, group ordering, wide-area behavior, source availability, and continuity sensitivity. These outputs should inform later production decisions but do not replace the canonical event/link layers.

In [ ]:
# Event intensity enrichment (peak/mean of each hazard's native source field)

if "single_hazard_events" not in globals() or "label_cubes" not in globals():
    raise RuntimeError("Run the event-labeling cell before this diagnostic.")

SOURCE_FIELD_VAR_NAMES = {
    "heatwave": "temp_da",
    "coldwave": "temp_da",
    "extreme_heat": "temp_hourly",
    "extreme_cold": "temp_hourly",
    "extreme_wind": "wind_da",
    "extreme_precip": "p_da",
    "hail": "hail_da",
    "lightning": "li_da",
    "space_weather_extreme": "e_mag",
    "wildfire": "wild_da",
}
SOURCE_FIELDS_FOR_INTENSITY = {hazard: globals()[var_name] for hazard, var_name in SOURCE_FIELD_VAR_NAMES.items() if var_name in globals()}


def _event_intensity(hazard_name, event_id):
    if hazard_name not in SOURCE_FIELDS_FOR_INTENSITY or hazard_name not in label_cubes:
        return np.nan, np.nan

    field_da = SOURCE_FIELDS_FOR_INTENSITY[hazard_name]
    labels_cube = label_cubes[hazard_name]
    mask_da = hazard_masks[hazard_name]
    event_num = int(event_id.rsplit("_", 1)[1])
    ti, yi, xi = np.where(labels_cube == event_num)
    if len(ti) == 0:
        return np.nan, np.nan

    mask_time = pd.DatetimeIndex(pd.to_datetime(mask_da["time"].values))
    field_time = pd.DatetimeIndex(pd.to_datetime(field_da["time"].values))
    field_idx = field_time.get_indexer(mask_time[ti])
    valid = field_idx >= 0
    if not valid.any():
        return np.nan, np.nan

    field_arr = np.asarray(field_da.values, dtype=float)
    values = field_arr[field_idx[valid], yi[valid], xi[valid]]
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan, np.nan
    return float(np.nanmax(values)), float(np.nanmean(values))


intensity_pairs = [_event_intensity(row["hazard"], row["event_id"]) for _, row in single_hazard_events.iterrows()]
single_hazard_events["peak_intensity"] = [pair[0] for pair in intensity_pairs]
single_hazard_events["mean_intensity"] = [pair[1] for pair in intensity_pairs]
single_hazard_events.to_csv(OUTPUT_DIR / "single_hazard_events_all.csv", index=False)

for hazard_name in event_tables:
    event_tables[hazard_name] = single_hazard_events[single_hazard_events["hazard"] == hazard_name].reset_index(drop=True)
    event_tables[hazard_name].to_csv(OUTPUT_DIR / f"single_hazard_events_{hazard_name}.csv", index=False)

print("Added peak_intensity/mean_intensity columns (native units per hazard's own source field; wildfire uses FRP/area score).")
display(single_hazard_events[["event_id", "hazard", "peak_intensity", "mean_intensity"]].head(10))

In [ ]:
# MYRIAD spatial-overlap sensitivity test
# Cross-hazard pairs require shared grid cells; temporal overlap remains optional within the lag.

if "event_cache_by_hazard" not in globals() or "single_hazard_events" not in globals():
    raise RuntimeError("Run the event-labeling/linking cell before this sensitivity test.")


def _events_are_linked_shared_cell_only(a, b, hazard_a, hazard_b):
    if a["start"] <= b["end"] and b["start"] <= a["end"]:
        gap_hours = 0.0
        lag_hours = _lag_hours_for_ordered_pair(hazard_a, hazard_b)
    elif b["start"] > a["end"]:
        gap_hours = float((b["start"] - a["end"]) / pd.Timedelta(hours=1))
        lag_hours = _lag_hours_for_ordered_pair(hazard_a, hazard_b)
    else:
        gap_hours = float((a["start"] - b["end"]) / pd.Timedelta(hours=1))
        lag_hours = _lag_hours_for_ordered_pair(hazard_b, hazard_a)

    if gap_hours > lag_hours:
        return False
    return bool(a["cells"] & b["cells"])


strict_edges = []
hazards_present = sorted(event_cache_by_hazard)
for i, hazard_a in enumerate(hazards_present):
    for hazard_b in hazards_present[i + 1:]:
        cache_a = event_cache_by_hazard[hazard_a]
        cache_b = event_cache_by_hazard[hazard_b]
        for event_id_a, event_a in cache_a.items():
            for event_id_b, event_b in cache_b.items():
                if _events_are_linked_shared_cell_only(event_a, event_b, hazard_a, hazard_b):
                    strict_edges.append({
                        "event_id_a": event_id_a,
                        "hazard_a": hazard_a,
                        "event_id_b": event_id_b,
                        "hazard_b": hazard_b,
                        "lag_hours": _lag_hours_for_ordered_pair(hazard_a, hazard_b),
                    })

strict_edges_df = pd.DataFrame(strict_edges)
strict_graph = nx_local.Graph()
strict_graph.add_nodes_from(single_hazard_events["event_id"])
strict_graph.add_edges_from((row["event_id_a"], row["event_id_b"]) for row in strict_edges)
strict_components = list(nx_local.connected_components(strict_graph))
strict_component_rows = []
for index, component in enumerate(strict_components, start=1):
    component_events = single_hazard_events[single_hazard_events["event_id"].isin(component)]
    strict_component_rows.append({
        "strict_mh_set_id": f"strict_mh_set_{index:06d}",
        "n_single_events": len(component_events),
        "n_hazard_types": component_events["hazard"].nunique(),
        "hazard_types": "|".join(sorted(component_events["hazard"].unique())),
        "start_time": component_events["start_time"].min(),
        "end_time": component_events["end_time"].max(),
        "duration_days": float((component_events["end_time"].max() - component_events["start_time"].min()) / pd.Timedelta(days=1) + 1.0),
    })

strict_component_summary = pd.DataFrame(strict_component_rows).sort_values("n_single_events", ascending=False)
strict_audit_dir = OUTPUT_ROOT / "diagnostics" / "linkage_audit" / f"spatial_overlap_only_{RUN_LABEL}"
strict_audit_dir.mkdir(parents=True, exist_ok=True)
strict_edges_df.to_csv(strict_audit_dir / "shared_cell_only_links.csv", index=False)
strict_component_summary.to_csv(strict_audit_dir / "shared_cell_only_components.csv", index=False)

print(f"Shared-cell-only accepted links: {len(strict_edges_df)}")
print(f"Shared-cell-only components: {len(strict_component_summary)}")
print("Largest shared-cell-only components:")
display(strict_component_summary.head(10))

In [ ]:
# MYRIAD hazard-group ordering and original-combination audit
# Groups are represented by chronological order of individual hazard events.

if "mh_members_df" not in globals() or "single_hazard_events" not in globals():
    raise RuntimeError("Run the event-labeling/linking cell before this group audit.")

paper_group_rows = []
for group_id, membership in mh_members_df.groupby("mh_set_id", sort=False):
    member_events = (
        membership[["event_id"]]
        .merge(single_hazard_events, on="event_id", how="left")
        .sort_values(["start_time", "end_time", "hazard", "event_id"])
        .reset_index(drop=True)
    )
    hazard_sequence = "|".join(member_events["hazard"].astype(str))
    unique_hazard_sequence = "|".join(dict.fromkeys(member_events["hazard"].astype(str)))
    paper_group_rows.append({
        "mh_set_id": group_id,
        "n_single_events": len(member_events),
        "n_hazard_types": member_events["hazard"].nunique(),
        "hazard_order": hazard_sequence,
        "unique_hazard_order": unique_hazard_sequence,
        "start_time": member_events["start_time"].min(),
        "end_time": member_events["end_time"].max(),
        "duration_days": float((member_events["end_time"].max() - member_events["start_time"].min()) / pd.Timedelta(days=1) + 1.0),
    })

paper_group_audit = pd.DataFrame(paper_group_rows)
combination_counts = (
    paper_group_audit.groupby("hazard_order", as_index=False)
    .size()
    .rename(columns={"size": "combination_frequency"})
    .sort_values("combination_frequency", ascending=False)
)
paper_group_audit = paper_group_audit.merge(combination_counts, on="hazard_order", how="left")

paper_group_dir = OUTPUT_ROOT / "diagnostics" / "linkage_audit" / f"paper_group_order_{RUN_LABEL}"
paper_group_dir.mkdir(parents=True, exist_ok=True)
paper_group_audit.to_csv(paper_group_dir / "hazard_groups_chronological_order.csv", index=False)
combination_counts.to_csv(paper_group_dir / "original_ordered_combinations.csv", index=False)

print(f"Chronologically ordered hazard groups: {len(paper_group_audit)}")
print(f"Original ordered hazard combinations: {len(combination_counts)}")
print("Most frequent ordered combinations:")
display(combination_counts.head(20))
display(paper_group_audit.sort_values("n_single_events", ascending=False).head(10))

In [ ]:
# Inspect the dominant ordered hazard group

if "paper_group_audit" not in globals():
    raise RuntimeError("Run the hazard-group ordering audit first.")

largest_group = paper_group_audit.sort_values("n_single_events", ascending=False).iloc[0]
full_sequence = largest_group["hazard_order"].split("|")
run_length_rows = []
for hazard_name in full_sequence:
    if run_length_rows and run_length_rows[-1]["hazard"] == hazard_name:
        run_length_rows[-1]["n_consecutive_events"] += 1
    else:
        run_length_rows.append({"hazard": hazard_name, "n_consecutive_events": 1})

run_length_sequence = " -> ".join(
    f"{row['hazard']} x{row['n_consecutive_events']}" if row["n_consecutive_events"] > 1 else row["hazard"]
    for row in run_length_rows
)

print(f"Group: {largest_group['mh_set_id']}")
print(f"Individual events: {largest_group['n_single_events']}")
print(f"Time span: {largest_group['start_time']} to {largest_group['end_time']}")
print("\nFull chronological sequence:")
print(" -> ".join(full_sequence))
print("\nRun-length sequence:")
print(run_length_sequence)
print("\nTransition counts:")
transition_counts = (
    pd.DataFrame({"from_hazard": full_sequence[:-1], "to_hazard": full_sequence[1:]})
    .value_counts(["from_hazard", "to_hazard"])
    .rename("count")
    .reset_index()
)
display(transition_counts.head(30))

In [ ]:
# Per-hazard spatial footprint and wide-area event audit

if "event_tables" not in globals() or "label_cubes" not in globals() or "hazard_masks" not in globals():
    raise RuntimeError("Run the event-labeling/linking cell before this audit.")

wide_area_rows = []
for hazard_name, event_table in event_tables.items():
    labels_array = label_cubes[hazard_name]
    mask_array = _safe_to_numpy(hazard_masks[hazard_name])
    grid_cell_count = int(labels_array.shape[1] * labels_array.shape[2])

    for _, event_row in event_table.iterrows():
        event_number = int(str(event_row["event_id"]).rsplit("_", 1)[1])
        event_voxels = labels_array == event_number
        event_times = np.where(event_voxels.any(axis=(1, 2)))[0]
        if len(event_times) == 0:
            continue

        footprint_cell_count = int(event_voxels.any(axis=0).sum())
        instantaneous_cell_counts = event_voxels[event_times].sum(axis=(1, 2))
        wide_area_rows.append({
            "event_id": event_row["event_id"],
            "hazard": hazard_name,
            "start_time": event_row["start_time"],
            "end_time": event_row["end_time"],
            "duration_days": event_row["duration_days"],
            "footprint_cells": footprint_cell_count,
            "footprint_fraction": footprint_cell_count / grid_cell_count,
            "max_instantaneous_cells": int(instantaneous_cell_counts.max()),
            "max_instantaneous_fraction": float(instantaneous_cell_counts.max() / grid_cell_count),
            "mean_instantaneous_fraction": float(instantaneous_cell_counts.mean() / grid_cell_count),
        })

wide_area_event_audit = pd.DataFrame(wide_area_rows)
wide_area_summary = (
    wide_area_event_audit.groupby("hazard", observed=True)
    .agg(
        n_events=("event_id", "size"),
        median_footprint_fraction=("footprint_fraction", "median"),
        p90_footprint_fraction=("footprint_fraction", lambda values: values.quantile(0.90)),
        median_max_instantaneous_fraction=("max_instantaneous_fraction", "median"),
        p90_max_instantaneous_fraction=("max_instantaneous_fraction", lambda values: values.quantile(0.90)),
        median_duration_days=("duration_days", "median"),
        wide_ge_10pct=("footprint_fraction", lambda values: int((values >= 0.10).sum())),
        wide_ge_25pct=("footprint_fraction", lambda values: int((values >= 0.25).sum())),
        wide_ge_50pct=("footprint_fraction", lambda values: int((values >= 0.50).sum())),
        max_footprint_fraction=("footprint_fraction", "max"),
    )
    .reset_index()
)
for threshold in ("wide_ge_10pct", "wide_ge_25pct", "wide_ge_50pct"):
    wide_area_summary[threshold.replace("wide_ge", "wide_rate")] = wide_area_summary[threshold] / wide_area_summary["n_events"]

wide_area_dir = OUTPUT_ROOT / "diagnostics" / "spatial_footprint_audit" / RUN_LABEL
wide_area_dir.mkdir(parents=True, exist_ok=True)
wide_area_event_audit.to_csv(wide_area_dir / "event_spatial_footprints.csv", index=False)
wide_area_summary.to_csv(wide_area_dir / "hazard_spatial_footprint_summary.csv", index=False)

print("Spatial footprint summary (fractions of the analysis grid):")
display(wide_area_summary.sort_values("n_events", ascending=False))
print("Largest events by footprint:")
display(wide_area_event_audit.sort_values("footprint_fraction", ascending=False).head(20))

In [ ]:
# May precipitation product context and current capability inventory

if "p_da" not in globals() or "ds_target_grid" not in globals():
    raise RuntimeError("Run the precipitation mask and grid-loading cells before this inventory.")

precipitation_product_context = {
    "source_product": "MRMS gauge-corrected 1-hour accumulated precipitation",
    "source_agency": "NOAA MRMS",
    "source_units": "mm",
    "source_purpose": "High-resolution spatial and short-term temporal precipitation estimation",
    "climate_comparison_product": "NCEI Contiguous U.S. Precipitation Rankings",
    "climate_comparison_basis": "nClimDiv/nClimGrid long-term climate dataset",
    "comparison_interpretation": "Not a direct validation target; products serve different purposes and should not be expected to agree numerically",
    "mrms_reference_url": "https://vlab.noaa.gov/web/wdtd/-/qpe-w-gauge-bias-correcti-1",
    "ncei_reference_url": "https://www.ncei.noaa.gov/access/monitoring/dyk/ranking-definition",
}

capability_inventory = pd.DataFrame([
    {"capability": "Single-hazard event detection", "current_status": "Primary validation focus", "current_evidence": f"8 active hazard types and {len(single_hazard_events)} May events", "main_limitation": "Non-wildfire event objects need independent physical and visual validation before frequency claims"},
    {"capability": "Wildfire event polygons", "current_status": "Highest confidence", "current_evidence": "Filtered GeoPackage inventory with final_area_km2 >= 5", "main_limitation": "Grid representation adds nearest-cell approximation for small polygons"},
    {"capability": "Space-weather event masks", "current_status": "Second-highest confidence", "current_evidence": "Operational 5-minute geoelectric-field aggregation, threshold, persistence, and wide-area diagnostics", "main_limitation": "Known source-data gaps remain; broad CONUS footprints require event-level review"},
    {"capability": "Gauge-corrected MRMS precipitation", "current_status": "Usable for short-term event detection; validation in progress", "current_evidence": "Complete May hourly coverage and physically plausible mm-scale values", "main_limitation": "Event segmentation and wide-area behavior need independent storm-case validation"},
    {"capability": "Hazard intensity indicators", "current_status": "Implemented", "current_evidence": "peak_intensity and mean_intensity saved per event in native source units", "main_limitation": "Units and physical meaning differ by hazard; no cross-hazard normalization yet"},
    {"capability": "Multi-hazard pair identification", "current_status": "Implemented for exploratory use", "current_evidence": "Pair links require spatial overlap and directional lag eligibility", "main_limitation": "Pair quality inherits the quality of each underlying event object"},
    {"capability": "Different temporal lags", "current_status": "Implemented for exploratory use", "current_evidence": "Directional PAIR_LAG_HOURS matrix plus 24-hour space-weather rule", "main_limitation": "Unlisted pairs use a 48-hour fallback and need explicit scientific review"},
    {"capability": "Ordered hazard combinations", "current_status": "Diagnostic only", "current_evidence": "Chronological hazard_order and unique_hazard_order are exported", "main_limitation": "Unconstrained graph connected components can create implausibly long sequences"},
    {"capability": "Hazard-group frequency", "current_status": "Not production-ready", "current_evidence": "Combination frequencies are counted from current graph components", "main_limitation": "Requires validated event objects and a transitive-closure/grouping rule"},
    {"capability": "Spatial hotspots", "current_status": "Not production-ready", "current_evidence": "Event centroids and footprints are available for pilot outputs", "main_limitation": "Requires coverage-normalized recurrence and combination-specific hotspot products"},
    {"capability": "Wide-area event diagnostics", "current_status": "Implemented", "current_evidence": "Cumulative and instantaneous footprint fractions are exported", "main_limitation": "Broad masks may be physical or segmentation artifacts depending on hazard"},
    {"capability": "Input coverage and cadence audit", "current_status": "Implemented", "current_evidence": "Full-period inventory, missing-date ranges, and representative cadence tables", "main_limitation": "Space-weather gaps remain in the current holdings"},
    {"capability": "Independent physical validation", "current_status": "Primary readiness gate", "current_evidence": "Wildfire polygon confidence, space-weather operational definition, and MRMS product context documented", "main_limitation": "Comparable case-based checks remain for precipitation, temperature, wind, drought, hail, and lightning"},
])

capability_dir = OUTPUT_ROOT / "diagnostics" / "capability_inventory" / RUN_LABEL
capability_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame([precipitation_product_context]).to_csv(capability_dir / "precipitation_product_context.csv", index=False)
capability_inventory.to_csv(capability_dir / "current_analysis_capability_inventory.csv", index=False)

print("Precipitation product context:")
print("  MRMS gauge-corrected 1-hour accumulation; units: mm")
print("  NCEI rankings are a separate long-term climate product, not a direct MRMS validation target")
print("Capability inventory:")
display(capability_inventory)

In [ ]:
# Non-wildfire event-object QA and manual-review queue

if "event_tables" not in globals() or "label_cubes" not in globals() or "hazard_masks" not in globals():
    raise RuntimeError("Run the event-labeling and intensity cells before event-object validation.")

non_wildfire_hazards = [hazard for hazard in HAZARD_RULES if hazard != "wildfire" and hazard in event_tables]
qa_rows = []
review_rows = []

for hazard_name in non_wildfire_hazards:
    event_table = event_tables[hazard_name]
    labels_array = label_cubes[hazard_name]
    mask_array = _safe_to_numpy(hazard_masks[hazard_name])
    source_available = hazard_name in SOURCE_FIELDS_FOR_INTENSITY
    source_field = SOURCE_FIELDS_FOR_INTENSITY.get(hazard_name)
    source_time_count = int(source_field.sizes.get("time", 0)) if source_field is not None else 0

    for _, event_row in event_table.iterrows():
        event_id = str(event_row["event_id"])
        event_number = int(event_id.rsplit("_", 1)[1])
        event_voxels = labels_array == event_number
        label_voxel_count = int(event_voxels.sum())
        label_time_count = int(event_voxels.any(axis=(1, 2)).sum())
        event_footprint_cells = int(event_voxels.any(axis=0).sum())
        event_peak = event_row.get("peak_intensity", np.nan)
        event_mean = event_row.get("mean_intensity", np.nan)
        qa_rows.append({
            "event_id": event_id,
            "hazard": hazard_name,
            "start_time": event_row["start_time"],
            "end_time": event_row["end_time"],
            "duration_days": event_row["duration_days"],
            "footprint_cells": event_footprint_cells,
            "footprint_fraction": event_footprint_cells / (labels_array.shape[1] * labels_array.shape[2]),
            "active_cell_steps": event_row["active_cell_steps"],
            "label_voxel_count": label_voxel_count,
            "label_time_count": label_time_count,
            "source_available": source_available,
            "source_time_count": source_time_count,
            "peak_intensity_finite": bool(np.isfinite(event_peak)),
            "mean_intensity_finite": bool(np.isfinite(event_mean)),
            "valid_time_order": bool(pd.Timestamp(event_row["start_time"]) <= pd.Timestamp(event_row["end_time"])),
            "valid_label_reference": bool(label_voxel_count > 0),
            "valid_footprint_filter": bool(event_footprint_cells >= MIN_EVENT_FOOTPRINT_CELLS.get(hazard_name, 1)),
            "valid_active_step_filter": bool(event_row["active_cell_steps"] >= MIN_EVENT_ACTIVE_CELL_STEPS.get(hazard_name, 1)),
        })

qa_event_table = pd.DataFrame(qa_rows)
qa_event_table["internal_qa_pass"] = qa_event_table[[
    "valid_time_order", "valid_label_reference", "valid_footprint_filter", "valid_active_step_filter"
]].all(axis=1)

for hazard_name in non_wildfire_hazards:
    hazard_rows = qa_event_table[qa_event_table["hazard"] == hazard_name].copy()
    if hazard_rows.empty:
        continue
    selected_ids = set()
    for sort_column, review_reason in [
        ("duration_days", "longest_duration"),
        ("footprint_fraction", "largest_cumulative_footprint"),
        ("active_cell_steps", "largest_active_cell_volume"),
    ]:
        selected = hazard_rows.sort_values(sort_column, ascending=False).head(2)
        for _, selected_row in selected.iterrows():
            if selected_row["event_id"] not in selected_ids:
                review_rows.append({**selected_row.to_dict(), "review_reason": review_reason})
                selected_ids.add(selected_row["event_id"])

    intensity_candidates = hazard_rows[hazard_rows["peak_intensity_finite"]].sort_values("event_id")
    if not intensity_candidates.empty:
        selected = intensity_candidates.sort_values("event_id").iloc[0]
        if selected["event_id"] not in selected_ids:
            review_rows.append({**selected.to_dict(), "review_reason": "intensity_reference_case"})

qa_summary = (
    qa_event_table.groupby("hazard", observed=True)
    .agg(
        n_events=("event_id", "size"),
        internal_qa_pass=("internal_qa_pass", "sum"),
        intensity_peak_available=("peak_intensity_finite", "sum"),
        intensity_mean_available=("mean_intensity_finite", "sum"),
        median_duration_days=("duration_days", "median"),
        p90_duration_days=("duration_days", lambda values: values.quantile(0.90)),
        max_duration_days=("duration_days", "max"),
        median_footprint_fraction=("footprint_fraction", "median"),
        p90_footprint_fraction=("footprint_fraction", lambda values: values.quantile(0.90)),
        max_footprint_fraction=("footprint_fraction", "max"),
    )
    .reset_index()
)
qa_summary["internal_qa_rate"] = qa_summary["internal_qa_pass"] / qa_summary["n_events"]
qa_summary["peak_intensity_rate"] = qa_summary["intensity_peak_available"] / qa_summary["n_events"]
qa_summary["mean_intensity_rate"] = qa_summary["intensity_mean_available"] / qa_summary["n_events"]

review_queue = pd.DataFrame(review_rows).drop_duplicates("event_id")
validation_dir = OUTPUT_ROOT / "diagnostics" / "event_object_validation" / RUN_LABEL
validation_dir.mkdir(parents=True, exist_ok=True)
qa_event_table.to_csv(validation_dir / "non_wildfire_event_object_qa.csv", index=False)
qa_summary.to_csv(validation_dir / "non_wildfire_event_object_qa_summary.csv", index=False)
review_queue.to_csv(validation_dir / "non_wildfire_manual_review_queue.csv", index=False)

print("Non-wildfire event-object QA summary:")
display(qa_summary)
print("Manual-review queue:")
display(review_queue.sort_values(["hazard", "review_reason", "event_id"]))
print(f"Saved event-object validation outputs: {validation_dir}")

In [ ]:
# Case-based review figures and movie for precipitation, hail, and lightning

if "event_tables" not in globals() or "label_cubes" not in globals() or "hazard_masks" not in globals():
    raise RuntimeError("Run the event-labeling cell before creating case-review outputs.")

from matplotlib import animation

review_hazards = [hazard for hazard in ["extreme_precip", "hail", "lightning"] if hazard in event_tables]
review_fields = {
    "extreme_precip": p_da,
    "hail": hail_da,
    "lightning": li_da,
}
review_dir = OUTPUT_ROOT / "diagnostics" / "event_case_review" / RUN_LABEL
review_dir.mkdir(parents=True, exist_ok=True)

review_case_rows = []
for hazard_name in review_hazards:
    event_table = event_tables[hazard_name]
    field_da = review_fields.get(hazard_name)
    if field_da is None or event_table.empty:
        continue

    field_times = pd.DatetimeIndex(pd.to_datetime(field_da["time"].values))
    labels_array = label_cubes[hazard_name]
    for _, event_row in event_table.sort_values("duration_days", ascending=False).head(5).iterrows():
        event_number = int(str(event_row["event_id"]).rsplit("_", 1)[1])
        event_voxels = labels_array == event_number
        event_time_indices = np.where(event_voxels.any(axis=(1, 2)))[0]
        if len(event_time_indices) == 0:
            continue
        event_time_indices = event_time_indices[event_time_indices < len(field_times)]
        if len(event_time_indices) == 0:
            continue

        event_values = np.asarray(field_da.values, dtype=float)[event_time_indices]
        event_spatial_max = np.nanmax(event_values.reshape(len(event_time_indices), -1), axis=1)
        peak_position = int(np.nanargmax(event_spatial_max))
        peak_index = int(event_time_indices[peak_position])
        review_case_rows.append({
            "event_id": event_row["event_id"],
            "hazard": hazard_name,
            "start_time": event_row["start_time"],
            "end_time": event_row["end_time"],
            "duration_days": event_row["duration_days"],
            "footprint_fraction": float(event_voxels.any(axis=0).mean()),
            "peak_source_value": float(event_spatial_max[peak_position]),
            "peak_source_time": field_times[peak_index],
        })

review_case_table = pd.DataFrame(review_case_rows)
review_case_table.to_csv(review_dir / "case_review_event_selection.csv", index=False)

lat_name = next((name for name in ["lat", "latitude", "y"] if name in p_da.coords), None)
lon_name = next((name for name in ["lon", "longitude", "x"] if name in p_da.coords), None)
if lat_name is None or lon_name is None:
    raise RuntimeError("Could not identify latitude/longitude coordinates for case-review figures.")
lat_grid = np.asarray(p_da[lat_name].values)
lon_grid = np.asarray(p_da[lon_name].values)

for _, case in review_case_table.iterrows():
    hazard_name = case["hazard"]
    field_da = review_fields[hazard_name]
    labels_array = label_cubes[hazard_name]
    event_number = int(str(case["event_id"]).rsplit("_", 1)[1])
    event_voxels = labels_array == event_number
    field_times = pd.DatetimeIndex(pd.to_datetime(field_da["time"].values))
    peak_index = int(np.argmin(np.abs(field_times - pd.Timestamp(case["peak_source_time"]))))
    field_values = np.asarray(field_da.values, dtype=float)
    event_peak_series = np.nanmax(field_values[event_voxels.any(axis=(1, 2))].reshape(-1, field_values.shape[-2] * field_values.shape[-1]), axis=1)
    event_time_indices = np.where(event_voxels.any(axis=(1, 2)))[0]
    event_time_indices = event_time_indices[event_time_indices < len(field_times)]
    event_times = field_times[event_time_indices]
    event_mask_at_peak = event_voxels[min(peak_index, labels_array.shape[0] - 1)]

    fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
    map_values = field_values[peak_index]
    image = axes[0, 0].pcolormesh(lon_grid, lat_grid, map_values, shading="auto", cmap="Blues")
    axes[0, 0].contour(lon_grid, lat_grid, event_mask_at_peak.astype(int), levels=[0.5], colors="red", linewidths=1.0)
    fig.colorbar(image, ax=axes[0, 0], label=f"{hazard_name} source units")
    axes[0, 0].set_title(f"Peak source field\n{case['peak_source_time']:%Y-%m-%d %H:%M UTC}")
    axes[0, 0].set_xlabel("Longitude")
    axes[0, 0].set_ylabel("Latitude")

    axes[0, 1].imshow(event_mask_at_peak, origin="lower", cmap="Reds", interpolation="none")
    axes[0, 1].set_title("Detected event footprint at peak")
    axes[0, 1].set_xlabel("Grid x")
    axes[0, 1].set_ylabel("Grid y")

    axes[1, 0].plot(event_times, event_peak_series[:len(event_times)], color="tab:red", linewidth=1.2)
    axes[1, 0].axvline(pd.Timestamp(case["peak_source_time"]), color="black", linestyle="--", linewidth=0.9)
    axes[1, 0].set_title("Event-footprint spatial maximum")
    axes[1, 0].set_ylabel("Source value")
    axes[1, 0].grid(alpha=0.25)

    axes[1, 1].axis("off")
    axes[1, 1].text(
        0.02, 0.95,
        "\n".join([
            f"Event: {case['event_id']}",
            f"Hazard: {hazard_name}",
            f"Start: {case['start_time']}",
            f"End: {case['end_time']}",
            f"Duration: {case['duration_days']:.2f} days",
            f"Cumulative footprint: {case['footprint_fraction']:.1%} of grid",
            f"Peak source value: {case['peak_source_value']:.3f}",
            "",
            "Review question:",
            "one physical event or multiple systems merged?",
        ]),
        va="top",
        fontsize=11,
    )
    fig.suptitle(f"Case review: {hazard_name}", fontsize=15)
    output_path = review_dir / f"{case['event_id']}_case_review.png"
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

# Month-view movie: raw fields and detected masks at a daily review cadence.
movie_hazards = [hazard for hazard in review_hazards if review_fields.get(hazard) is not None]
movie_times = pd.date_range(run_start.normalize(), run_end.normalize(), freq="D")
movie_path = review_dir / f"precip_hail_lightning_review_{RUN_LABEL}.mp4"
fig, axes = plt.subplots(len(movie_hazards), 2, figsize=(12, 4 * len(movie_hazards)), squeeze=False, constrained_layout=True)
movie_images = []
movie_masks = []
movie_titles = []
for row_index, hazard_name in enumerate(movie_hazards):
    field_da = review_fields[hazard_name]
    field_times = pd.DatetimeIndex(pd.to_datetime(field_da["time"].values))
    field_index = int(np.argmin(np.abs(field_times - movie_times[0])))
    field_values = np.asarray(field_da.values, dtype=float)
    labels_array = label_cubes[hazard_name]
    mask_times = pd.DatetimeIndex(pd.to_datetime(hazard_masks[hazard_name]["time"].values))
    mask_index = int(np.argmin(np.abs(mask_times - movie_times[0])))
    raw_image = axes[row_index, 0].pcolormesh(lon_grid, lat_grid, field_values[field_index], shading="auto", cmap="Blues")
    mask_image = axes[row_index, 1].imshow(labels_array[mask_index] > 0, origin="lower", cmap="Reds", interpolation="none", vmin=0, vmax=1)
    axes[row_index, 0].set_title(f"{hazard_name} source")
    axes[row_index, 1].set_title(f"{hazard_name} detected mask")
    movie_images.append((hazard_name, raw_image, field_values, field_times))
    movie_masks.append((hazard_name, mask_image, labels_array, mask_times))
    movie_titles.append(axes[row_index, 0].set_ylabel(""))


def update_review_movie(frame_number):
    timestamp = movie_times[frame_number]
    artists = []
    for row_index, (hazard_name, raw_image, field_values, field_times) in enumerate(movie_images):
        field_index = int(np.argmin(np.abs(field_times - timestamp)))
        raw_image.set_array(field_values[field_index].ravel())
        axes[row_index, 0].set_title(f"{hazard_name} source | {field_times[field_index]:%Y-%m-%d %H:%M UTC}")
        artists.append(raw_image)
    for row_index, (hazard_name, mask_image, labels_array, mask_times) in enumerate(movie_masks):
        mask_index = int(np.argmin(np.abs(mask_times - timestamp)))
        mask_image.set_data((labels_array[mask_index] > 0).astype(np.uint8))
        axes[row_index, 1].set_title(f"{hazard_name} detected mask | active cells: {int((labels_array[mask_index] > 0).sum())}")
        artists.append(mask_image)
    fig.suptitle(f"May 2024 event-object review | {timestamp:%Y-%m-%d}", fontsize=15)
    return artists

review_movie = animation.FuncAnimation(fig, update_review_movie, frames=len(movie_times), interval=350, blit=False)
review_movie.save(movie_path, writer=animation.FFMpegWriter(fps=3, codec="libx264", bitrate=3000), dpi=120)
plt.close(fig)

print(f"Saved case-review table and figures: {review_dir}")
print(f"Saved review movie: {movie_path}")
display(review_case_table)

In [ ]:
# Diagnose the May precipitation benchmark discrepancy

precip_values = np.asarray(p_da.values, dtype=float)
monthly_values = np.asarray(monthly_precip_in.values, dtype=float)
valid_fraction_by_cell = np.isfinite(precip_values).mean(axis=0)
monthly_valid = np.isfinite(monthly_values)

unweighted_mean_in = float(np.nanmean(monthly_values))
median_cell_total_in = float(np.nanmedian(monthly_values))
p90_cell_total_in = float(np.nanpercentile(monthly_values, 90))
max_cell_total_in = float(np.nanmax(monthly_values))
missing_sample_fraction = float(1.0 - np.isfinite(precip_values).mean())
valid_cell_fraction = float(monthly_valid.mean())

precip_metadata_diagnostic = pd.DataFrame([{
    "source_variable": precip_var,
    "dims": str(p_da.dims),
    "shape": str(p_da.shape),
    "raw_units_attribute": p_da.attrs.get("units", "missing"),
    "raw_long_name": p_da.attrs.get("long_name", p_da.attrs.get("description", "missing")),
    "raw_min": float(np.nanmin(precip_values)),
    "raw_max": float(np.nanmax(precip_values)),
    "raw_mean_nonzero": float(np.nanmean(precip_values[precip_values > 0])) if np.any(precip_values > 0) else np.nan,
    "missing_sample_fraction": missing_sample_fraction,
    "valid_monthly_cell_fraction": valid_cell_fraction,
    "unweighted_monthly_mean_in": unweighted_mean_in,
    "area_weighted_monthly_mean_in": analysis_monthly_mean_in,
    "median_cell_monthly_total_in": median_cell_total_in,
    "p90_cell_monthly_total_in": p90_cell_total_in,
    "max_cell_monthly_total_in": max_cell_total_in,
    "noaa_reference_total_in": NOAA_MAY_2024_PCP_IN,
    "noaa_to_analysis_ratio": NOAA_MAY_2024_PCP_IN / analysis_monthly_mean_in,
}])

precip_metadata_diagnostic.to_csv(capability_dir / "may_2024_precipitation_metadata_diagnostic.csv", index=False)
print("May precipitation source metadata and coverage:")
display(precip_metadata_diagnostic)
print("Precipitation source attributes:")
print(dict(p_da.attrs))

In [ ]:
# Within-hazard continuity sensitivity: remove cross-time spatial drift

if "hazard_masks" not in globals():
    raise RuntimeError("Run the hazard-mask construction cell before this sensitivity test.")


def _same_time_spatial_structure():
    structure = np.zeros((3, 3, 3), dtype=bool)
    structure[1, 1, 1] = True
    structure[1, 0, 1] = True
    structure[1, 2, 1] = True
    structure[1, 1, 0] = True
    structure[1, 1, 2] = True
    structure[0, 1, 1] = True
    structure[2, 1, 1] = True
    return structure


def _label_sensitivity_summary(mask_da, hazard_name, structure):
    mask_array = _safe_to_numpy(mask_da)
    bridged_array = _bridge_short_time_gaps(mask_array, int(EVENT_END_GAP_STEPS.get(hazard_name, 0)))
    labels_array, n_labels = ndimage.label(bridged_array, structure=structure)
    rows = []
    if n_labels == 0:
        return rows

    times = pd.DatetimeIndex(pd.to_datetime(mask_da["time"].values))
    if len(times) > 1:
        time_step_hours = float(np.nanmedian(np.diff(times) / np.timedelta64(1, "h")))
    else:
        time_step_hours = 24.0
    min_cells = int(MIN_EVENT_FOOTPRINT_CELLS.get(hazard_name, 1))
    min_active_steps = int(MIN_EVENT_ACTIVE_CELL_STEPS.get(hazard_name, 1))

    for event_number in range(1, n_labels + 1):
        event_voxels = labels_array == event_number
        event_times = np.where(event_voxels.any(axis=(1, 2)))[0]
        if len(event_times) == 0:
            continue
        footprint_cells = int(event_voxels.any(axis=0).sum())
        active_cell_steps = int(event_voxels.sum())
        if active_cell_steps < min_active_steps or footprint_cells < min_cells:
            continue

        start_time = times[event_times[0]]
        end_time = times[event_times[-1]]
        duration_hours = float((end_time - start_time) / pd.Timedelta(hours=1) + time_step_hours)
        duration_days = duration_hours / 24.0
        if TEMPORAL_MODE.get(hazard_name, "daily") == "daily" and duration_days < float(MIN_EVENT_DURATION_DAYS):
            continue

        rows.append({
            "hazard": hazard_name,
            "event_number": event_number,
            "start_time": start_time,
            "end_time": end_time,
            "duration_days": duration_days,
            "footprint_cells": footprint_cells,
            "footprint_fraction": footprint_cells / (labels_array.shape[1] * labels_array.shape[2]),
            "active_cell_steps": active_cell_steps,
        })
    return rows


continuity_test_hazards = [
    hazard for hazard in [
        "extreme_precip", "hail", "extreme_heat", "extreme_cold", "extreme_wind",
        "drought_proxy", "heatwave", "coldwave", "lightning", "space_weather_extreme",
    ] if hazard in hazard_masks
]
strict_continuity_rows = []
strict_structure = _same_time_spatial_structure()
for hazard_name in continuity_test_hazards:
    strict_continuity_rows.extend(_label_sensitivity_summary(hazard_masks[hazard_name], hazard_name, strict_structure))

strict_continuity_events = pd.DataFrame(strict_continuity_rows)
current_continuity_events = wide_area_event_audit[[
    "event_id", "hazard", "start_time", "end_time", "duration_days", "footprint_cells",
    "footprint_fraction", "mean_instantaneous_fraction",
]].copy()

continuity_summary_rows = []
for hazard_name in continuity_test_hazards:
    current = current_continuity_events[current_continuity_events["hazard"] == hazard_name]
    strict = strict_continuity_events[strict_continuity_events["hazard"] == hazard_name]
    continuity_summary_rows.append({
        "hazard": hazard_name,
        "current_events": len(current),
        "strict_events": len(strict),
        "event_count_ratio_strict_current": len(strict) / len(current) if len(current) else np.nan,
        "current_median_duration_days": current["duration_days"].median() if len(current) else np.nan,
        "strict_median_duration_days": strict["duration_days"].median() if len(strict) else np.nan,
        "current_max_duration_days": current["duration_days"].max() if len(current) else np.nan,
        "strict_max_duration_days": strict["duration_days"].max() if len(strict) else np.nan,
        "current_max_footprint_fraction": current["footprint_fraction"].max() if len(current) else np.nan,
        "strict_max_footprint_fraction": strict["footprint_fraction"].max() if len(strict) else np.nan,
        "current_events_ge_25pct": int((current["footprint_fraction"] >= 0.25).sum()) if len(current) else 0,
        "strict_events_ge_25pct": int((strict["footprint_fraction"] >= 0.25).sum()) if len(strict) else 0,
    })

continuity_sensitivity_summary = pd.DataFrame(continuity_summary_rows)
continuity_test_dir = OUTPUT_ROOT / "diagnostics" / "within_hazard_continuity" / RUN_LABEL
continuity_test_dir.mkdir(parents=True, exist_ok=True)
strict_continuity_events.to_csv(continuity_test_dir / "strict_same_cell_temporal_events.csv", index=False)
continuity_sensitivity_summary.to_csv(continuity_test_dir / "continuity_sensitivity_summary.csv", index=False)

print("Current versus strict same-cell-temporal continuity:")
display(continuity_sensitivity_summary)
print("Largest strict events by cumulative footprint:")
display(strict_continuity_events.sort_values("footprint_fraction", ascending=False).head(20))

In [ ]:
# Pipeline output summary

if "single_hazard_events" not in globals() or "mh_sets_df" not in globals():
    raise RuntimeError("Run Cells 1-4 before viewing the pipeline summary.")

print(f"Analysis window: {run_start:%Y-%m-%d} to {run_end:%Y-%m-%d}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Single-hazard events: {len(single_hazard_events)}")
print(f"Multi-hazard event sets: {len(mh_sets_df)}")

summary = (
    single_hazard_events.groupby("hazard", observed=True)
    .agg(
        n_events=("event_id", "size"),
        first_start=("start_time", "min"),
        last_end=("end_time", "max"),
        median_footprint_cells=("footprint_cells", "median"),
    )
    .reindex(list(HAZARD_RULES))
    .fillna({"n_events": 0})
    .reset_index()
)
summary["n_events"] = summary["n_events"].astype(int)
display(summary)

if not mh_sets_df.empty:
    display(mh_sets_df.sort_values(["start_time", "mh_set_id"]).head(20))

## Optional Development Diagnostics: Rendering and Case Review

These cells generate summaries, maps, movies, and case-review artifacts. They are not required to create the database layers.

In [ ]:
# Space-weather event and driver diagnostic movie

import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import ListedColormap

SPACE_HAZARD = "space_weather_extreme"
MOVIE_FRAME_STRIDE = 3  # Three 5-minute samples per frame: a 15-minute display cadence.
MOVIE_FPS = 8
OMNI_CSV = Path("outputs") / "myriad_conus_2024_05" / "omni_drivers_2024_05.csv"
MOVIE_OUT = OUTPUT_DIR / f"space_weather_drivers_{RUN_LABEL}_conus.mp4"
COUNTY_SHP = Path("_cache") / "tl_2024_us_county" / "tl_2024_us_county.shp"

if SPACE_HAZARD not in hazard_masks or SPACE_HAZARD not in label_cubes:
    raise RuntimeError("Run the mask and event-linking cells before creating this movie.")
if not OMNI_CSV.exists():
    raise FileNotFoundError(f"OMNI driver CSV not found: {OMNI_CSV}")
if not animation.writers.is_available("ffmpeg"):
    raise RuntimeError("ffmpeg is required to export the MP4 diagnostic.")

sw_mask = hazard_masks[SPACE_HAZARD]
labels = label_cubes[SPACE_HAZARD]
times = pd.DatetimeIndex(pd.to_datetime(sw_mask.time.values))
window = np.where((times >= run_start) & (times < run_end + pd.Timedelta(days=1)))[0]
if len(window) == 0:
    raise RuntimeError("No operational space-weather timestamps fall in the configured window.")
frame_indices = window[::MOVIE_FRAME_STRIDE]

sw_events = single_hazard_events[single_hazard_events["hazard"] == SPACE_HAZARD]
valid_labels = np.array([int(event_id.rsplit("_", 1)[1]) for event_id in sw_events["event_id"]], dtype=int)
if valid_labels.size == 0:
    raise RuntimeError("No filtered space-weather events were identified for this run.")

rng = np.random.default_rng(42)
shuffled_labels = valid_labels.copy()
rng.shuffle(shuffled_labels)
color_index = {label: index + 1 for index, label in enumerate(shuffled_labels)}
colors = np.vstack([[0, 0, 0, 0], plt.get_cmap("turbo", len(valid_labels))(np.arange(len(valid_labels)))])
event_cmap = ListedColormap(colors)

omni = pd.read_csv(OMNI_CSV, parse_dates=["time"]).sort_values("time")
omni = omni[(omni["time"] >= run_start) & (omni["time"] < run_end + pd.Timedelta(days=1))]
if omni.empty:
    raise RuntimeError("The OMNI driver file has no samples in the configured window.")

lat = np.asarray(sw_mask["lat"].values)
lon = np.asarray(sw_mask["lon"].values)
state_boundaries = None
if COUNTY_SHP.exists() and gpd is not None:
    counties = gpd.read_file(COUNTY_SHP)
    state_boundaries = counties[~counties["STATEFP"].isin({"02", "15", "60", "66", "69", "72", "78"})].dissolve(by="STATEFP").boundary

fig = plt.figure(figsize=(14, 10), constrained_layout=True)
grid = fig.add_gridspec(4, 1, height_ratios=[1, 1, 1, 4])
axes = [fig.add_subplot(grid[index]) for index in range(3)]
map_ax = fig.add_subplot(grid[3])
driver_specs = [("Bz_GSM_nT", "Bz GSM [nT]", "tab:blue"), ("AE_nT", "AE [nT]", "tab:red"), ("SymH_nT", "SYM-H [nT]", "tab:purple")]
markers = []
for ax, (column, ylabel, color) in zip(axes, driver_specs):
    ax.plot(omni["time"], omni[column], color=color, linewidth=0.65)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    markers.append(ax.axvline(times[frame_indices[0]], color="black", linewidth=1.2))
axes[-1].set_xlabel("UTC")

if state_boundaries is not None:
    state_boundaries.plot(ax=map_ax, color="0.35", linewidth=0.45, zorder=2)
map_ax.pcolormesh(lon, lat, np.zeros_like(labels[0]), shading="auto", facecolor="none", edgecolors="0.80", linewidth=0.12, zorder=1)
map_ax.set_xlabel("Longitude")
map_ax.set_ylabel("Latitude")
map_ax.set_aspect("equal", adjustable="box")


def frame_colors(label_slice):
    frame = np.zeros_like(label_slice, dtype=np.int32)
    for label in np.unique(label_slice):
        if label in color_index:
            frame[label_slice == label] = color_index[label]
    return frame

first_frame = frame_colors(labels[frame_indices[0]])
event_mesh = map_ax.pcolormesh(lon, lat, first_frame, shading="auto", cmap=event_cmap, vmin=0, vmax=len(valid_labels), zorder=3)
title = map_ax.set_title("")


def update(frame_number):
    index = frame_indices[frame_number]
    frame_time = times[index]
    for marker in markers:
        marker.set_xdata([frame_time, frame_time])
    event_mesh.set_array(frame_colors(labels[index]).ravel())
    active = np.unique(labels[index][np.isin(labels[index], valid_labels)]).size
    title.set_text(f"Operational space-weather events | {frame_time:%Y-%m-%d %H:%M UTC} | active events: {active}")
    return [event_mesh, title, *markers]

movie = animation.FuncAnimation(fig, update, frames=len(frame_indices), interval=1000 / MOVIE_FPS, blit=False)
movie.save(MOVIE_OUT, writer=animation.FFMpegWriter(fps=MOVIE_FPS, codec="libx264", bitrate=2600), dpi=140)
plt.close(fig)

print(f"Saved: {MOVIE_OUT}")
print(f"Frames: {len(frame_indices)} | display cadence: {MOVIE_FRAME_STRIDE * 5} min | duration: {len(frame_indices) / MOVIE_FPS:.1f} s")

In [ ]:
# All-hazard event-mask overview movie

from matplotlib import animation
from matplotlib.colors import ListedColormap

OVERVIEW_FRAME_STRIDE = 6  # 30-minute display cadence from the 5-minute space-weather timeline.
OVERVIEW_FPS = 8
OVERVIEW_OUT = OUTPUT_DIR / f"all_hazard_events_{RUN_LABEL}_conus.mp4"
OVERVIEW_COUNTY_SHP = Path("_cache") / "tl_2024_us_county" / "tl_2024_us_county.shp"

if "label_cubes" not in globals() or "single_hazard_events" not in globals():
    raise RuntimeError("Run the mask and event-linking cells before creating the overview movie.")
if not animation.writers.is_available("ffmpeg"):
    raise RuntimeError("ffmpeg is required to export the MP4 overview.")

frame_hazard = "space_weather_extreme"
frame_times = pd.DatetimeIndex(pd.to_datetime(hazard_masks[frame_hazard].time.values))
window_indices = np.where((frame_times >= run_start) & (frame_times < run_end + pd.Timedelta(days=1)))[0]
if len(window_indices) == 0:
    raise RuntimeError("No space-weather timestamps exist in the configured analysis window.")
frame_indices = window_indices[::OVERVIEW_FRAME_STRIDE]
render_times = frame_times[frame_indices]
hazard_order = list(HAZARD_RULES)

lat = np.asarray(hazard_masks[frame_hazard]["lat"].values)
lon = np.asarray(hazard_masks[frame_hazard]["lon"].values)
lon_limits = (float(np.nanmin(lon)), float(np.nanmax(lon)))
lat_limits = (float(np.nanmin(lat)), float(np.nanmax(lat)))

state_boundaries = None
if OVERVIEW_COUNTY_SHP.exists() and gpd is not None:
    counties = gpd.read_file(OVERVIEW_COUNTY_SHP)
    state_boundaries = counties[~counties["STATEFP"].isin({"02", "15", "60", "66", "69", "72", "78"})].dissolve(by="STATEFP").boundary

valid_labels_by_hazard = {}
color_index_by_hazard = {}
cmap_by_hazard = {}
for hazard in hazard_order:
    events = single_hazard_events[single_hazard_events["hazard"] == hazard]
    labels = np.array([int(event_id.rsplit("_", 1)[1]) for event_id in events["event_id"]], dtype=int)
    valid_labels_by_hazard[hazard] = labels
    shuffled = labels.copy()
    np.random.default_rng(abs(hash(hazard)) % (2**32)).shuffle(shuffled)
    color_index_by_hazard[hazard] = {label: index + 1 for index, label in enumerate(shuffled)}
    colors = np.vstack([[0, 0, 0, 0], plt.get_cmap("turbo", max(len(labels), 1))(np.arange(max(len(labels), 1)))])
    cmap_by_hazard[hazard] = ListedColormap(colors)


def event_slice_at_time(hazard, timestamp):
    mask_time = pd.DatetimeIndex(pd.to_datetime(hazard_masks[hazard].time.values))
    if TEMPORAL_MODE.get(hazard) == "daily":
        matches = np.where(mask_time.normalize() == timestamp.normalize())[0]
    else:
        matches = np.where(mask_time == timestamp)[0]
    if len(matches) == 0:
        return np.zeros_like(label_cubes[hazard][0], dtype=np.int32)
    return label_cubes[hazard][int(matches[0])]


def colored_events(hazard, timestamp):
    label_slice = event_slice_at_time(hazard, timestamp)
    result = np.zeros_like(label_slice, dtype=np.int32)
    label_map = color_index_by_hazard[hazard]
    for label in np.unique(label_slice):
        if label in label_map:
            result[label_slice == label] = label_map[label]
    return result

fig, axes = plt.subplots(3, 3, figsize=(15, 9), constrained_layout=True)
meshes = {}
titles = {}
for ax, hazard in zip(axes.flat, hazard_order):
    if state_boundaries is not None:
        state_boundaries.plot(ax=ax, color="0.35", linewidth=0.45, zorder=2)
    initial = colored_events(hazard, render_times[0])
    mesh = ax.pcolormesh(lon, lat, initial, shading="auto", cmap=cmap_by_hazard[hazard], vmin=0, vmax=max(len(valid_labels_by_hazard[hazard]), 1), linewidth=0, zorder=3)
    ax.set_xlim(*lon_limits)
    ax.set_ylim(*lat_limits)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([])
    ax.set_yticks([])
    titles[hazard] = ax.set_title(hazard.replace("_", " "), fontsize=10)
    meshes[hazard] = mesh


def update(frame_number):
    timestamp = render_times[frame_number]
    artists = []
    for hazard in hazard_order:
        labels_now = event_slice_at_time(hazard, timestamp)
        meshes[hazard].set_array(colored_events(hazard, timestamp).ravel())
        active_filtered = np.unique(labels_now[np.isin(labels_now, valid_labels_by_hazard[hazard])]).size
        titles[hazard].set_text(f"{hazard.replace('_', ' ')} | events: {active_filtered}")
        artists.extend([meshes[hazard], titles[hazard]])
    fig.suptitle(f"CONUS filtered hazard-event masks | {timestamp:%Y-%m-%d %H:%M UTC}", fontsize=14)
    return artists

movie = animation.FuncAnimation(fig, update, frames=len(render_times), interval=1000 / OVERVIEW_FPS, blit=False)
movie.save(OVERVIEW_OUT, writer=animation.FFMpegWriter(fps=OVERVIEW_FPS, codec="libx264", bitrate=3500), dpi=120)
plt.close(fig)

print(f"Saved: {OVERVIEW_OUT}")
print(f"Frames: {len(render_times)} | display cadence: {OVERVIEW_FRAME_STRIDE * 5} min | duration: {len(render_times) / OVERVIEW_FPS:.1f} s")